# GNN Pattern Learner


## 0 · Imports
Everything the modules below need — third-party (PyTorch, PyG, pyTigerGraph, pandas, numpy) and the standard library. Inlined code shares this one namespace.

In [1]:
# stdlib
import math, random, time, re, json, os, warnings
from abc import abstractmethod
from collections.abc import Callable, Iterable, Iterator, Mapping, Sequence
from dataclasses import dataclass, field
from enum import Enum, IntEnum, auto
from pathlib import Path
from typing import Any, Protocol, cast, runtime_checkable

# third-party
import numpy as np
import pandas as pd
import torch
from torch import Tensor
from torch.nn import Embedding, Linear, Module, ModuleDict
from torch.optim import AdamW

# PyTorch Geometric
import torch_geometric
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATv2Conv, HeteroConv
from torch_geometric.sampler import NodeSamplerInput
from torch_geometric.typing import EdgeType, NodeType

# pyTigerGraph
import pyTigerGraph as tg
from numpy.typing import NDArray

print('torch', torch.__version__, '| torch_geometric', torch_geometric.__version__)

/Users/abrahamchandy/Documents/Work/companies/2026/TigerGraph/MulePatternLearner/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.12.0 | torch_geometric 2.7.0


## 1 · Schema enums
`schema/enums.py`

The vocabulary of the graph: every **vertex type** (Account, Party, Resolved_Entity, …) and **edge type** (HAS_PAID, Account_Account, Party_Entity, …) as Python enums. Everything downstream refers to these names instead of raw strings, so a typo is a `NameError`, not a silent miss.

In [2]:
from enum import StrEnum


class VertexType(StrEnum):
    ACCOUNT = "Account"
    PARTY = "Party"
    DEVICE = "Device"
    IP = "IP"
    PHONE = "Phone"
    EMAIL = "Email"
    NAME = "Name"
    BIRTHDATE = "Birthdate"
    STREET_ADDRESS = "Street_Address"


class EdgeType(StrEnum):
    HAS_PAID = "HAS_PAID"
    REV_HAS_PAID = "reverse_HAS_PAID"
    ACCOUNT_ACCOUNT = "Account_Account"
    PARTY_HAS_ACCOUNT = "Party_Has_Account"
    HAS_DEVICE = "Has_Device"
    HAS_IP = "Has_IP"
    HAS_PHONE = "Has_Phone"
    HAS_EMAIL = "Has_Email"
    HAS_NAME = "Has_Name"
    HAS_BIRTHDATE = "Has_Birthdate"
    HAS_STREET_ADDRESS = "Has_Street_Address"


UNDIRECTED_EDGES: frozenset[str] = frozenset(
    {
        EdgeType.ACCOUNT_ACCOUNT.value,
        EdgeType.PARTY_HAS_ACCOUNT.value,
        EdgeType.HAS_DEVICE.value,
        EdgeType.HAS_IP.value,
        EdgeType.HAS_PHONE.value,
        EdgeType.HAS_EMAIL.value,
        EdgeType.HAS_NAME.value,
        EdgeType.HAS_BIRTHDATE.value,
        EdgeType.HAS_STREET_ADDRESS.value,
    }
)

NODE_TYPES: tuple[str, ...] = tuple(vt.value for vt in VertexType)

## 2 · Schema specs — the PyG / pyTigerGraph bridge
`schema/specs.py`

Declares each edge's direction and which vertex/edge **features** pyTigerGraph should export. The `pytigergraph_*` and `pyg_metadata` helpers are the single place the graph schema is translated into the tensors PyG expects — node feature lists, edge feature lists, and the `(node_types, edge_types)` metadata the model is built from.

In [4]:
"""
Schema declarations for the graph, and their conversion into PyG's loader spec.

Defines which attributes are node features (per vertex type), which edges exist
(EdgeSpec / EDGE_SPECS, including synthetic reverses for undirected edges), and
which attributes are labels: pu_label (the partially-known training target) and
is_fraud (eval-only ground truth, never a feature).

The pytigergraph_* functions translate these declarations into the v_in_feats /
v_out_labels / e_in_feats dicts the loader consumes.
"""

from dataclasses import dataclass
from collections.abc import Mapping


TARGET_VERTEX: str = VertexType.ACCOUNT.value

TARGET_LABEL_ATTR: str = "pu_label"
EVAL_LABEL_ATTR: str = "is_fraud"

SPLIT_ATTRS: tuple[str, ...] = ("is_train", "is_val", "is_test")

AUX_LABEL_ATTRS: tuple[str, ...] = ("fraud", "victim")

ACCOUNT_EXTRA_ATTRS: tuple[str, ...] = SPLIT_ATTRS + AUX_LABEL_ATTRS + (EVAL_LABEL_ATTR,)

ACCOUNT_NUMERIC_FEATURES: tuple[str, ...] = (
    "pagerank",
    "com_size",
    "aa_degree",
    "triangle_count",
    "clustering_coef",
    "in_degree",
    "out_degree",
    "in_amount",
    "out_amount",
    "in_txn_count",
    "out_txn_count",
    "fan_in_ratio",
    "fan_out_ratio",
    "pass_through_ratio",
    "net_flow",
    "activity_span_days",
    "days_since_last_txn",
    "account_age_days",
    "mean_inter_txn_days",
    "txn_per_active_day",
    "burst_ratio",
    "active_bin_count",
    "activity_concentration",
    "peak_bin_fraction",
    "early_late_ratio",
    "in_out_lag_days",
    "device_share_cnt",
    "ip_share_cnt",
    "phone_share_cnt",
    "email_share_cnt",
    "is_external",
)


LEAKY_EXCLUDED_ATTRS: tuple[str, ...] = (
    "is_fraud",
    "pu_label",
    "fraud",
    "victim",
    "mule_cnt",
    "fraud_ip",
    "fraud_device",
    "trans_in_mule_ratio",
    "trans_out_mule_ratio",
    "shortest_path_length",
    "com_id",
)

DEFERRED_CATEGORICAL_ATTRS: tuple[str, ...] = ("external_type", "category")


VERTEX_FEATURES: Mapping[str, tuple[str, ...]] = {
    VertexType.ACCOUNT.value: ACCOUNT_NUMERIC_FEATURES,
    VertexType.PARTY.value: (),
    VertexType.DEVICE.value: ("is_blocked",),
    VertexType.IP.value: ("is_blocked",),
    VertexType.PHONE.value: (),
    VertexType.EMAIL.value: (),
    VertexType.NAME.value: (),
    VertexType.BIRTHDATE.value: (),
    VertexType.STREET_ADDRESS.value: (),
}


@dataclass(frozen=True, slots=True)
class EdgeSpec:
    """One directed PyG relation: (src, name, dst), where name is the TG edge id.

    raw_attrs:         TG edge attributes pulled into edge_attr (pre-transform).
    has_time_attrs:    trailing DATETIME attrs need epoch->feature conversion.
    synthetic_reverse: a reverse direction we add for an undirected TG edge.
    """

    src: str
    name: str
    dst: str
    raw_attrs: tuple[str, ...] = ()
    has_time_attrs: bool = False
    synthetic_reverse: bool = False

    @property
    def triple(self) -> tuple[str, str, str]:
        return (self.src, self.name, self.dst)


_A = VertexType.ACCOUNT
_P = VertexType.PARTY
_DEV = VertexType.DEVICE
_IP = VertexType.IP
_PH = VertexType.PHONE
_EM = VertexType.EMAIL
_NM = VertexType.NAME
_BD = VertexType.BIRTHDATE
_SA = VertexType.STREET_ADDRESS

_HAS_PAID_ATTRS = (
    "total_amount",
    "total_num_txns",
    "first_txn_date",
    "last_txn_date",
    "span_days",
)

HAS_PAID_SEQUENCE_ATTRS: tuple[str, ...] = ("amount_bins", "count_bins")

HAS_PAID_MAX_BINS: int = 100


def _spec(
    src: VertexType,
    name: EdgeType,
    dst: VertexType,
    attrs: tuple[str, ...] = (),
    time: bool = False,
    syn: bool = False,
) -> EdgeSpec:
    return EdgeSpec(src.value, name.value, dst.value, attrs, time, syn)


_BASE_EDGE_SPECS: tuple[EdgeSpec, ...] = (
    _spec(_A, EdgeType.HAS_PAID, _A, _HAS_PAID_ATTRS, time=True),
    _spec(_A, EdgeType.REV_HAS_PAID, _A, _HAS_PAID_ATTRS, time=True),
    _spec(_A, EdgeType.ACCOUNT_ACCOUNT, _A, ("weight",)),
    _spec(_P, EdgeType.PARTY_HAS_ACCOUNT, _A),
    _spec(_P, EdgeType.HAS_DEVICE, _DEV),
    _spec(_P, EdgeType.HAS_IP, _IP),
    _spec(_P, EdgeType.HAS_PHONE, _PH),
    _spec(_P, EdgeType.HAS_EMAIL, _EM),
    _spec(_P, EdgeType.HAS_NAME, _NM),
    _spec(_P, EdgeType.HAS_BIRTHDATE, _BD),
    _spec(_P, EdgeType.HAS_STREET_ADDRESS, _SA),
)


def _expand_undirected(base: tuple[EdgeSpec, ...]) -> tuple[EdgeSpec, ...]:
    """Add synthetic-reverse triples for undirected TG edges (src != dst)."""
    out: list[EdgeSpec] = list(base)
    for s in base:
        if s.name not in UNDIRECTED_EDGES or s.src == s.dst:
            continue
        out.append(EdgeSpec(s.dst, s.name, s.src, s.raw_attrs, s.has_time_attrs, True))
    return tuple(out)


EDGE_SPECS: tuple[EdgeSpec, ...] = _expand_undirected(_BASE_EDGE_SPECS)


def edge_spec(triple: tuple[str, str, str]) -> EdgeSpec:
    for s in EDGE_SPECS:
        if s.triple == triple:
            return s
    raise KeyError(f"Unknown edge triple {triple!r}")


def pytigergraph_v_in_feats() -> dict[str, list[str]]:
    return {vt: list(attrs) for vt, attrs in VERTEX_FEATURES.items()}


def pytigergraph_v_out_labels() -> dict[str, list[str]]:
    """The training target: pu_label (the masked, realistic PU label)."""
    return {TARGET_VERTEX: [TARGET_LABEL_ATTR]}


def pytigergraph_v_extra_feats() -> dict[str, list[str]]:
    """Carried alongside seeds: split masks + aux labels + is_fraud (eval only).
    pu_label is the target (v_out_labels), so it is not repeated here."""
    return {TARGET_VERTEX: list(ACCOUNT_EXTRA_ATTRS)}


def pytigergraph_e_in_feats() -> dict[str, list[str]]:
    """Scalar edge attributes per TG edge-type name (the fixed-width features
    that become PyG edge_attr directly). Reverses reuse the same name, so
    de-dup by name. Does NOT include the HAS_PAID bin sequences â those are
    variable-length LISTs fetched separately (see e_sequence_feats)."""
    out: dict[str, list[str]] = {}
    for s in EDGE_SPECS:
        if s.raw_attrs and s.name not in out:
            out[s.name] = list(s.raw_attrs)
    return out


def pytigergraph_e_sequence_feats() -> dict[str, list[str]]:
    """Variable-length LIST edge attributes per TG edge-type name. The loader
    pulls these and pads/truncates each to HAS_PAID_MAX_BINS to form a dense
    per-edge sequence tensor (consumed flat or by a sequence encoder). Kept
    separate from e_in_feats because they need different handling than scalar
    edge_attr."""
    return {EdgeType.HAS_PAID.value: list(HAS_PAID_SEQUENCE_ATTRS)}


def pyg_metadata() -> tuple[list[str], list[tuple[str, str, str]]]:
    return list(VERTEX_FEATURES.keys()), [s.triple for s in EDGE_SPECS]

## 3 · Edge feature width
`features/edge_spec.py`

The one piece of arithmetic the whole edge pipeline hangs on: a HAS_PAID edge is described by **8 scalar features** plus a **time-series of bins** (2 channels: amount + count). `flat_edge_dim(max_bins) = 8 + max_bins × 2`. With 13 bins that's **34** — the model's `edge_dim`.

In [5]:
"""
Feature-width definitions for HAS_PAID edges.

Sets the dimension of each edge's feature vector: 8 scalar features plus 2
binned features (amount, count) unrolled across max_bins time-slices.
"""


class EdgeFeatureError(ValueError):
    pass


SCALAR_FEATURE_NAMES: tuple[str, ...] = (
    "amount_transferred",
    "transaction_count",
    "active_span_days",
    "active_bin_count",
    "recency_days",
    "duration_days",
    "active_bin_fraction",
    "amount_per_transaction",
)

NUM_SCALAR_FEATURES: int = len(SCALAR_FEATURE_NAMES)
NUM_BIN_CHANNELS: int = 2


def flat_edge_dim(max_bins: int) -> int:
    """Width of the per-edge feature vector per bin count."""
    if max_bins < 1:
        raise EdgeFeatureError(f"max_bins must be >= 1, got {max_bins}")
    return NUM_SCALAR_FEATURES + max_bins * NUM_BIN_CHANNELS

## 4 · Account node features (the 31)
`features/nodes.py`

Turns a raw TigerGraph Account row into the model's **31-feature** input vector. Each feature has a declared **transform** (log1p for heavy-tailed counts, symlog for signed flows, clamp-sentinel for missing values, boolean for flags). `build_account_features` applies them; `NodeFeatures` is the validated `[N, 31]` tensor; `FeatureNormalizer` standardizes columns (fit on train, applied everywhere).

In [6]:
import math
from collections.abc import Sequence
from dataclasses import dataclass
from enum import Enum, auto
from typing import cast

import torch
from torch import Tensor



class NodeFeatureError(ValueError):
    pass


class Transform(Enum):
    """How a raw scalar feature is mapped before entering the model."""

    LOG1P = auto()
    SYMLOG = auto()
    IDENTITY = auto()
    BOOLEAN = auto()
    CLAMP_SENTINEL = auto()


_MISSING_SENTINEL: float = -1.0


def log1p_compress(value: float) -> float:
    """Compress a heavy-tailed non-negative quantity via log(1 + x)."""
    return math.log1p(max(value, 0.0))


def symlog_compress(value: float) -> float:
    """Sign-preserving log compression for signed heavy-tailed values.

    Computes sign(x) * log(1 + |x|): continuous through zero (0 -> 0), keeps
    the sign, and reduces to log1p_compress for non-negative inputs. Preferred
    over sign(x) * log(|x|), which is undefined at zero and amplifies near-zero
    noise.
    """
    sign = 1.0 if value >= 0.0 else -1.0
    return sign * math.log1p(abs(value))


_ACCOUNT_FEATURE_TRANSFORMS: dict[str, Transform] = {
    "pagerank": Transform.IDENTITY,
    "com_size": Transform.LOG1P,
    "aa_degree": Transform.LOG1P,
    "triangle_count": Transform.LOG1P,
    "clustering_coef": Transform.IDENTITY,
    "in_degree": Transform.LOG1P,
    "out_degree": Transform.LOG1P,
    "in_amount": Transform.LOG1P,
    "out_amount": Transform.LOG1P,
    "in_txn_count": Transform.LOG1P,
    "out_txn_count": Transform.LOG1P,
    "fan_in_ratio": Transform.IDENTITY,
    "fan_out_ratio": Transform.IDENTITY,
    "pass_through_ratio": Transform.IDENTITY,
    "net_flow": Transform.SYMLOG,
    "activity_span_days": Transform.IDENTITY,
    "days_since_last_txn": Transform.CLAMP_SENTINEL,
    "account_age_days": Transform.CLAMP_SENTINEL,
    "mean_inter_txn_days": Transform.CLAMP_SENTINEL,
    "txn_per_active_day": Transform.IDENTITY,
    "burst_ratio": Transform.IDENTITY,
    "active_bin_count": Transform.LOG1P,
    "activity_concentration": Transform.IDENTITY,
    "peak_bin_fraction": Transform.IDENTITY,
    "early_late_ratio": Transform.SYMLOG,
    "in_out_lag_days": Transform.SYMLOG,
    "device_share_cnt": Transform.LOG1P,
    "ip_share_cnt": Transform.LOG1P,
    "phone_share_cnt": Transform.LOG1P,
    "email_share_cnt": Transform.LOG1P,
    "is_external": Transform.BOOLEAN,
}


def account_feature_names() -> tuple[str, ...]:
    """Account feature names in tensor-column order (matches schema spec)."""
    return ACCOUNT_NUMERIC_FEATURES


def _validate_transform_coverage() -> None:
    spec = set(ACCOUNT_NUMERIC_FEATURES)
    mapped = set(_ACCOUNT_FEATURE_TRANSFORMS)
    missing = spec - mapped
    extra = mapped - spec
    if missing or extra:
        raise NodeFeatureError(
            "account transform policy out of sync with ACCOUNT_NUMERIC_FEATURES: "
            + f"missing={sorted(missing)}, unexpected={sorted(extra)}"
        )


_validate_transform_coverage()

NUM_ACCOUNT_FEATURES: int = len(ACCOUNT_NUMERIC_FEATURES)


@dataclass(frozen=True, slots=True)
class NodeFeatures:
    node_ids: tuple[str, ...]
    feats: Tensor
    feature_names: tuple[str, ...]

    @property
    def num_nodes(self) -> int:
        return len(self.node_ids)

    def __post_init__(self) -> None:
        n = len(self.node_ids)
        f = len(self.feature_names)
        if tuple(self.feats.shape) != (n, f):
            raise NodeFeatureError(f"feats shape {tuple(self.feats.shape)} != ({n}, {f})")


def _as_float(value: object, field: str) -> float:
    if isinstance(value, bool):
        return 1.0 if value else 0.0
    if isinstance(value, (int, float)):
        return float(value)
    raise NodeFeatureError(f"{field}: expected number, got {type(value).__name__}")


def _apply_transform(raw: float, transform: Transform) -> float:
    if transform is Transform.LOG1P:
        return log1p_compress(raw)
    if transform is Transform.SYMLOG:
        return symlog_compress(raw)
    if transform is Transform.BOOLEAN:
        return 1.0 if raw != 0.0 else 0.0
    if transform is Transform.CLAMP_SENTINEL:
        return 0.0 if raw == _MISSING_SENTINEL else raw
    return raw


def _vertex_attributes(vertex: object) -> tuple[str, dict[str, object]]:
    if not isinstance(vertex, dict):
        raise NodeFeatureError(f"vertex is not a dict: {vertex!r}")
    vertex_typed: dict[str, object] = {
        str(k): v for k, v in cast(dict[object, object], vertex).items()
    }
    node_id = vertex_typed.get("v_id")
    if not isinstance(node_id, str) or not node_id:
        raise NodeFeatureError(f"vertex missing 'v_id': {vertex_typed!r}")
    attrs = vertex_typed.get("attributes")
    if not isinstance(attrs, dict):
        raise NodeFeatureError(f"vertex missing attributes dict: {vertex_typed!r}")
    attrs_typed: dict[str, object] = {
        str(k): v for k, v in cast(dict[object, object], attrs).items()
    }
    return node_id, attrs_typed


def _build_account_row(attrs: dict[str, object]) -> list[float]:
    row: list[float] = []
    for name in ACCOUNT_NUMERIC_FEATURES:
        raw = _as_float(attrs.get(name, 0.0), name)
        row.append(_apply_transform(raw, _ACCOUNT_FEATURE_TRANSFORMS[name]))
    return row


def build_account_features(vertices: Sequence[object]) -> NodeFeatures:
    node_ids: list[str] = []
    rows: list[float] = []

    for vertex in vertices:
        node_id, attrs = _vertex_attributes(vertex)
        node_ids.append(node_id)
        rows.extend(_build_account_row(attrs))

    n = len(node_ids)
    feats = torch.tensor(rows, dtype=torch.float32).reshape(n, NUM_ACCOUNT_FEATURES)

    return NodeFeatures(
        node_ids=tuple(node_ids),
        feats=feats,
        feature_names=ACCOUNT_NUMERIC_FEATURES,
    )


@dataclass(frozen=True, slots=True)
class FeatureNormalizer:
    """Per-feature z-score, applied after the log/symlog transforms.

    Fit on the TRAIN split only (then reused unchanged for val/test/inference),
    or val/test statistics leak into training. Saved with the checkpoint so
    scoring reuses the exact training-time standardization.
    """

    mean: Tensor
    std: Tensor

    def __post_init__(self) -> None:
        if self.mean.shape != (NUM_ACCOUNT_FEATURES,) or self.std.shape != (NUM_ACCOUNT_FEATURES,):
            raise NodeFeatureError(
                f"normalizer mean/std must be ({NUM_ACCOUNT_FEATURES},); "
                + f"got {tuple(self.mean.shape)} / {tuple(self.std.shape)}"
            )

    def apply(self, feats: Tensor) -> Tensor:
        return (feats - self.mean.to(feats.device)) / self.std.to(feats.device)


def normalizer_from_features(feats: Tensor) -> FeatureNormalizer:
    """Fit a FeatureNormalizer from an (n, NUM_ACCOUNT_FEATURES) feature matrix.

    Caller is responsible for passing TRAIN-split features only. A small floor on
    std keeps constant / near-constant columns from producing huge or NaN values.
    """
    if feats.ndim != 2 or feats.shape[1] != NUM_ACCOUNT_FEATURES:
        raise NodeFeatureError(
            f"expected (n, {NUM_ACCOUNT_FEATURES}) features; got {tuple(feats.shape)}"
        )
    mean = feats.mean(dim=0)
    std = feats.std(dim=0, unbiased=False).clamp_min(1e-6)
    return FeatureNormalizer(mean=mean, std=std)

## 5 · Edge features (scalars + bins)
`features/temporal.py`

The HAS_PAID counterpart. Parses each payment edge's timestamps and pre-aggregated bins into `EdgeFeatures`: **scalar_feats [E, 8]** and **bin_seq [E, bins, 2]**. `flat_edge_features` then flattens that into the `[E, 34]` `edge_attr` the attention layer consumes.

In [7]:
import math
import warnings
from collections.abc import Sequence
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import cast

import torch
from torch import Tensor


_SECONDS_PER_DAY: float = 86400.0


def _string_keyed(mapping: dict[object, object]) -> dict[str, object]:
    return {str(k): v for k, v in mapping.items()}


_NEAR_CAP_WARN_FRACTION: float = 0.9


def log1p_compress(value: float) -> float:
    """Compress a heavy-tailed non-negative quantity via log(1 + x).

    Monetary amounts and transaction counts span many orders of magnitude;
    raw values let the largest edges dominate. log(1 + x) compresses that
    range, and the (1 + x) shift maps 0 -> 0 (a bin with no activity stays
    zero) while remaining defined at x = 0. Negative inputs are clamped to 0.
    """
    return math.log1p(max(value, 0.0))


@dataclass(frozen=True, slots=True)
class EdgeFeatures:
    src_ids: tuple[str, ...]
    dst_ids: tuple[str, ...]
    scalar_feats: Tensor
    bin_seq: Tensor
    max_bins: int

    @property
    def num_edges(self) -> int:
        return len(self.src_ids)

    def __post_init__(self) -> None:
        if self.max_bins < 1:
            raise EdgeFeatureError(f"max_bins must be >= 1, got {self.max_bins}")
        e = len(self.src_ids)
        if len(self.dst_ids) != e:
            raise EdgeFeatureError(f"src/dst length mismatch: {e} vs {len(self.dst_ids)}")
        if tuple(self.scalar_feats.shape) != (e, NUM_SCALAR_FEATURES):
            raise EdgeFeatureError(
                f"scalar_feats shape {tuple(self.scalar_feats.shape)} "
                + f"!= ({e}, {NUM_SCALAR_FEATURES})"
            )
        if tuple(self.bin_seq.shape) != (e, self.max_bins, NUM_BIN_CHANNELS):
            raise EdgeFeatureError(
                f"bin_seq shape {tuple(self.bin_seq.shape)} "
                + f"!= ({e}, {self.max_bins}, {NUM_BIN_CHANNELS})"
            )


def _as_float(value: object, field: str) -> float:
    if isinstance(value, bool):
        raise EdgeFeatureError(f"{field}: expected number, got bool")
    if isinstance(value, (int, float)):
        return float(value)
    raise EdgeFeatureError(f"{field}: expected number, got {type(value).__name__}")


def _as_int(value: object, field: str) -> int:
    if isinstance(value, bool):
        raise EdgeFeatureError(f"{field}: expected int, got bool")
    if isinstance(value, int):
        return value
    if isinstance(value, float) and value.is_integer():
        return int(value)
    raise EdgeFeatureError(f"{field}: expected int, got {type(value).__name__}")


_DATETIME_FORMAT: str = "%Y-%m-%d %H:%M:%S"


def _as_epoch_seconds(value: object, field: str) -> float:
    """Parse a DATETIME field into epoch seconds (UTC).

    TigerGraph serializes DATETIME differently across configurations: an
    installed-query path may return epoch-second integers, while other setups
    return a formatted "YYYY-MM-DD HH:MM:SS" string. Both are accepted; strings
    are interpreted as UTC so the result is deterministic regardless of the
    machine's local timezone.
    """
    if isinstance(value, bool):
        raise EdgeFeatureError(f"{field}: expected datetime, got bool")
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        try:
            parsed = datetime.strptime(value, _DATETIME_FORMAT)
        except ValueError as exc:
            raise EdgeFeatureError(
                f"{field}: could not parse datetime string {value!r} "
                + f"with format {_DATETIME_FORMAT!r}"
            ) from exc
        return parsed.replace(tzinfo=timezone.utc).timestamp()
    raise EdgeFeatureError(
        f"{field}: expected epoch number or datetime string, " + f"got {type(value).__name__}"
    )


def _attributes_of(edge: object) -> dict[str, object]:
    if not isinstance(edge, dict):
        raise EdgeFeatureError(f"edge is not a dict: {edge!r}")
    edge_typed = _string_keyed(cast(dict[object, object], edge))
    attrs = edge_typed.get("attributes")
    if not isinstance(attrs, dict):
        raise EdgeFeatureError(f"edge missing attributes dict: {edge_typed!r}")
    return _string_keyed(cast(dict[object, object], attrs))


@dataclass(frozen=True, slots=True)
class _RawScalars:
    total_amount: float
    total_num_txns: float
    span_days: float
    num_bins: int
    first_epoch_s: float
    last_epoch_s: float


@dataclass(frozen=True, slots=True)
class _TimeDeltas:
    recency_days: float
    duration_days: float


def _extract_endpoints(edge: dict[str, object]) -> tuple[str, str]:
    """Pull the source and destination account ids from an edge object."""
    src = edge.get("from_id")
    dst = edge.get("to_id")
    if not isinstance(src, str) or not src:
        raise EdgeFeatureError(f"edge missing 'from_id': {edge!r}")
    if not isinstance(dst, str) or not dst:
        raise EdgeFeatureError(f"edge missing 'to_id': {edge!r}")
    return src, dst


def _parse_scalar_fields(attrs: dict[str, object]) -> _RawScalars:
    """Extract and type-check the raw scalar attributes of one edge."""
    return _RawScalars(
        total_amount=_as_float(attrs.get("total_amount"), "total_amount"),
        total_num_txns=_as_float(attrs.get("total_num_txns"), "total_num_txns"),
        span_days=_as_float(attrs.get("span_days"), "span_days"),
        num_bins=_as_int(attrs.get("num_bins"), "num_bins"),
        first_epoch_s=_as_epoch_seconds(attrs.get("first_txn_date"), "first_txn_date"),
        last_epoch_s=_as_epoch_seconds(attrs.get("last_txn_date"), "last_txn_date"),
    )


def _compute_time_deltas(raw: _RawScalars, reference_epoch_s: float) -> _TimeDeltas:
    """Convert epoch-second timestamps into day-scale recency and duration.

    recency_days is measured from a caller-supplied reference time (the
    prediction/snapshot moment), never 'now', so the transform is leakage-safe.
    """
    recency_days = max((reference_epoch_s - raw.last_epoch_s) / _SECONDS_PER_DAY, 0.0)
    duration_days = max((raw.last_epoch_s - raw.first_epoch_s) / _SECONDS_PER_DAY, 0.0)
    return _TimeDeltas(recency_days=recency_days, duration_days=duration_days)


def _pad_bins(raw: object, field: str, max_bins: int) -> tuple[list[float], int]:
    """Pad or truncate a bin list to exactly max_bins; report the raw length."""
    if not isinstance(raw, list):
        raise EdgeFeatureError(f"{field}: expected list, got {type(raw).__name__}: {raw!r}")
    raw_list: list[object] = list(cast(list[object], raw))
    raw_len = len(raw_list)
    out: list[float] = [0.0] * max_bins
    n = min(raw_len, max_bins)
    for i in range(n):
        out[i] = _as_float(raw_list[i], f"{field}[{i}]")
    return out, raw_len


def _count_active_bins(count_bins: list[float]) -> int:
    """Count time bins that contain at least one transaction (count > 0).

    This is the true per-edge temporal footprint, derived from the data, not
    the global num_bins attribute (which is identical on every edge and
    therefore carries no per-edge signal).
    """
    return sum(1 for c in count_bins if c > 0.0)


def _build_scalar_row(
    raw: _RawScalars, deltas: _TimeDeltas, active_bin_count: int, max_bins: int
) -> list[float]:
    """Assemble the per-edge scalar feature vector in SCALAR_FEATURE_NAMES order."""
    active_bin_fraction = active_bin_count / max_bins if max_bins > 0 else 0.0
    amount_per_transaction = (
        raw.total_amount / raw.total_num_txns if raw.total_num_txns > 0.0 else 0.0
    )
    row: list[float] = [
        log1p_compress(raw.total_amount),
        log1p_compress(raw.total_num_txns),
        raw.span_days,
        float(active_bin_count),
        deltas.recency_days,
        deltas.duration_days,
        active_bin_fraction,
        amount_per_transaction,
    ]
    if len(row) != NUM_SCALAR_FEATURES:
        raise EdgeFeatureError(
            f"internal: built {len(row)} scalars, expected {NUM_SCALAR_FEATURES}"
        )
    return row


def _build_bin_row(amount_bins: list[float], count_bins: list[float], max_bins: int) -> list[float]:
    """Interleave (compressed amount, count) per bin into a flat row of 2*max_bins."""
    interleaved: list[float] = []
    for i in range(max_bins):
        interleaved.append(log1p_compress(amount_bins[i]))
        interleaved.append(max(count_bins[i], 0.0))
    return interleaved


def _warn_on_bin_capacity(
    max_raw_bins: int, truncated_count: int, total_edges: int, max_bins: int
) -> None:
    if truncated_count > 0:
        warnings.warn(
            f"{truncated_count}/{total_edges} edges had more than "
            + f"max_bins={max_bins} and were SILENTLY TRUNCATED "
            + f"(largest seen: {max_raw_bins}). Real temporal data was dropped. "
            + "Re-derive max_bins from the data and rebuild.",
            stacklevel=2,
        )
        return
    near_cap_floor = int(_NEAR_CAP_WARN_FRACTION * max_bins)
    if max_bins > 0 and near_cap_floor <= max_raw_bins < max_bins:
        warnings.warn(
            f"Largest edge had {max_raw_bins} bins, approaching max_bins={max_bins}. "
            + "Consider re-deriving the width before bins exceed it.",
            stacklevel=2,
        )


def build_edge_features(
    edges: Sequence[object],
    reference_epoch_s: float,
    max_bins: int,
) -> EdgeFeatures:
    if max_bins < 1:
        raise EdgeFeatureError(f"max_bins must be >= 1, got {max_bins}")

    src_ids: list[str] = []
    dst_ids: list[str] = []
    scalar_rows: list[float] = []
    bin_rows: list[float] = []

    max_raw_bins = 0
    truncated_count = 0

    for edge in edges:
        if not isinstance(edge, dict):
            raise EdgeFeatureError(f"edge is not a dict: {edge!r}")
        edge_typed = _string_keyed(cast(dict[object, object], edge))
        attrs = _attributes_of(edge_typed)

        src, dst = _extract_endpoints(edge_typed)
        raw = _parse_scalar_fields(attrs)
        deltas = _compute_time_deltas(raw, reference_epoch_s)

        amount_bins, amount_len = _pad_bins(attrs.get("amount_bins"), "amount_bins", max_bins)
        count_bins, count_len = _pad_bins(attrs.get("count_bins"), "count_bins", max_bins)
        raw_bin_len = max(amount_len, count_len)

        active_bin_count = _count_active_bins(count_bins)

        src_ids.append(src)
        dst_ids.append(dst)
        scalar_rows.extend(_build_scalar_row(raw, deltas, active_bin_count, max_bins))
        bin_rows.extend(_build_bin_row(amount_bins, count_bins, max_bins))

        if raw_bin_len > max_raw_bins:
            max_raw_bins = raw_bin_len
        if raw_bin_len > max_bins:
            truncated_count += 1

    _warn_on_bin_capacity(max_raw_bins, truncated_count, len(src_ids), max_bins)

    e = len(src_ids)
    scalar_feats = torch.tensor(scalar_rows, dtype=torch.float32).reshape(e, NUM_SCALAR_FEATURES)
    bin_seq = torch.tensor(bin_rows, dtype=torch.float32).reshape(e, max_bins, NUM_BIN_CHANNELS)

    return EdgeFeatures(
        src_ids=tuple(src_ids),
        dst_ids=tuple(dst_ids),
        scalar_feats=scalar_feats,
        bin_seq=bin_seq,
        max_bins=max_bins,
    )


def flatten_bin_seq(bin_seq: Tensor) -> Tensor:
    """Collapse a [E, bins, channels] sequence into a flat [E, bins*channels]."""
    if bin_seq.dim() != 3:
        raise EdgeFeatureError(
            f"bin_seq must be 3D [E, bins, channels], got {tuple(bin_seq.shape)}"
        )
    e = bin_seq.shape[0]
    return bin_seq.reshape(e, -1)


def flat_edge_features(features: EdgeFeatures) -> Tensor:
    """Concatenate scalar features and flattened bins into one [E, k] matrix.

    Some GNN layers (those taking a fixed edge feature dimension) require edge
    attributes as a flat 2D tensor rather than a 3D sequence. k == flat_edge_dim.
    """
    return torch.cat([features.scalar_feats, flatten_bin_seq(features.bin_seq)], dim=1)

## 6 · Device select
`device.py`

Tiny helper: pick CUDA / MPS / CPU.

In [8]:
import torch


def select_device() -> torch.device:
    """Pick the best available accelerator: CUDA, then Apple MPS, then CPU.

    Returns a torch.device the caller moves the model and every batch onto. The
    order reflects throughput for batched tensor math; CPU is the portable
    fallback. torch.backends.mps.is_available() returns False on non-macOS
    builds, so the MPS branch is simply skipped there.
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

## 7 · PU masking
`data/pu_masking.py`

How the **positive-unlabelled** labels are made. Known mules become positives (`pu_label=1`); **everyone else is unlabelled, not innocent** (`pu_label=0`). This is the labelling rule the whole nnPU approach exists to respect.

In [9]:
from collections.abc import Sequence
from dataclasses import dataclass
from enum import IntEnum
import random


class Bucket(IntEnum):
    """Per-account masking bucket."""

    UNLABELED_NEG = 0
    REVEALED_POS = 1
    HIDDEN_POS = 2


@dataclass(frozen=True, slots=True)
class MaskConfig:
    """Parameters for ring-aware PU masking.

    reveal_prevalence:
        Target fraction of ALL mules that remain labeled (pu_label = 1).
        Default 0.10 models "about 10% of fraudsters are known". Raised from an
        earlier 0.04: at 4% the train/val split was starved of positives (val
        could round to zero revealed positives, which the trainer rejects), and
        a near-saturated ROC-AUC over so few positives is noisy to select on.
        More revealed positives stabilise the selection signal and give nnPU's
        positive-risk term cleaner gradients. Still a realistic minority.
    dark_ring_fraction:
        Fraction of mule-containing rings made FULLY DARK (no revealed labels).
        Their accounts are emitted as forced-test (see MaskResult.forced_test).
        Lowered 0.30 -> 0.20 so more positive-bearing rings stay available to
        train/val rather than being pinned to test with zero training signal.
    seed:
        RNG seed; the assignment is deterministic given inputs and seed.
    """

    reveal_prevalence: float = 0.10
    dark_ring_fraction: float = 0.20
    seed: int = 1337

    def validate(self) -> None:
        if not (0.0 < self.reveal_prevalence <= 1.0):
            raise ValueError("reveal_prevalence must be in (0, 1].")
        if not (0.0 <= self.dark_ring_fraction <= 1.0):
            raise ValueError("dark_ring_fraction must be in [0, 1].")


@dataclass(frozen=True, slots=True)
class MaskRecord:
    """Per-account input for masking: identity, truth, and ring membership."""

    account_id: str
    is_mule: bool
    ring_id: int


@dataclass(frozen=True, slots=True)
class MaskResult:
    """Per-account masking outputs, aligned to the input order.

    forced_test is the set of dark-ring account ids that must be held out of
    training. It is the explicit handoff to the splitter: masking decides which
    rings are fully dark, and the caller passes forced_test to split_accounts.
    Masking itself performs no train/val/test assignment.
    """

    account_ids: tuple[str, ...]
    pu_label: tuple[int, ...]
    true_label: tuple[int, ...]
    bucket: tuple[int, ...]
    forced_test: frozenset[str]

    def summary(self) -> dict[str, int]:
        revealed = sum(1 for b in self.bucket if b == Bucket.REVEALED_POS)
        hidden = sum(1 for b in self.bucket if b == Bucket.HIDDEN_POS)
        neg = sum(1 for b in self.bucket if b == Bucket.UNLABELED_NEG)
        return {
            "accounts": len(self.account_ids),
            "true_mules": revealed + hidden,
            "revealed_positives": revealed,
            "hidden_positives": hidden,
            "unlabeled_negatives": neg,
            "forced_test_accounts": len(self.forced_test),
        }


def _ring_to_mule_indices(
    records: Sequence[MaskRecord],
) -> dict[int, list[int]]:
    rings: dict[int, list[int]] = {}
    for i, r in enumerate(records):
        if r.ring_id > 0 and r.is_mule:
            rings.setdefault(r.ring_id, []).append(i)
    return rings


def resolve_account_rings(
    txn_endpoints: Sequence[tuple[str, int]],
) -> dict[str, int]:
    out: dict[str, int] = {}
    for account_id, ring_id in txn_endpoints:
        if account_id not in out:
            out[account_id] = 0
        if ring_id > 0 and out[account_id] == 0:
            out[account_id] = ring_id
    return out


def apply_pu_mask(records: Sequence[MaskRecord], config: MaskConfig) -> MaskResult:
    """
    Simulate realistic, clustered fraud-label discovery on synthetic data.

    Algorithm:
      1. Identify mule-containing rings. Choose dark_ring_fraction of them to be
         fully dark (no revealed labels). Their accounts are emitted in
         forced_test (to be pinned to TEST by the splitter), so the model never
         trains on a fully-undiscovered ring.
      2. From the NON-dark mules, reveal a random subset sized so the global
         revealed count is about reveal_prevalence * total_mules.
    """
    config.validate()
    rng = random.Random(config.seed)

    n = len(records)
    total_mules = sum(1 for r in records if r.is_mule)

    pu: list[int] = [0] * n
    true: list[int] = [1 if r.is_mule else 0 for r in records]
    bucket: list[int] = [int(Bucket.UNLABELED_NEG)] * n

    ring_mules = _ring_to_mule_indices(records)
    ring_ids = sorted(ring_mules.keys())
    rng.shuffle(ring_ids)
    n_dark = int(round(config.dark_ring_fraction * len(ring_ids)))
    dark_rings: set[int] = set(ring_ids[:n_dark])

    dark_mule_indices: set[int] = set()
    for rid in dark_rings:
        for idx in ring_mules[rid]:
            dark_mule_indices.add(idx)

    forced_test: set[str] = {
        records[i].account_id for i, r in enumerate(records) if r.ring_id in dark_rings
    }

    target_revealed = int(round(config.reveal_prevalence * total_mules))
    revealable: list[int] = [
        i for i, r in enumerate(records) if r.is_mule and i not in dark_mule_indices
    ]
    rng.shuffle(revealable)
    revealed_set: set[int] = set(revealable[:target_revealed])

    for i, r in enumerate(records):
        if r.is_mule:
            if i in revealed_set:
                pu[i] = 1
                bucket[i] = int(Bucket.REVEALED_POS)
            else:
                pu[i] = 0
                bucket[i] = int(Bucket.HIDDEN_POS)
        else:
            pu[i] = 0
            bucket[i] = int(Bucket.UNLABELED_NEG)

    return MaskResult(
        account_ids=tuple(r.account_id for r in records),
        pu_label=tuple(pu),
        true_label=tuple(true),
        bucket=tuple(bucket),
        forced_test=frozenset(forced_test),
    )

## 8 · Train / val / test split
`data/splitting.py`

Splits accounts into train/val/test **by party**, stratifying the rare positives across splits so each split sees some — and so related accounts never straddle the split boundary (a leakage guard).

In [10]:
from collections.abc import Sequence
from dataclasses import dataclass
from enum import IntEnum
import random


class Split(IntEnum):
    TRAIN = 0
    VAL = 1
    TEST = 2


@dataclass(frozen=True, slots=True)
class SplitConfig:
    """
    Parameters for a party-grouped train/val/test split.

    val_fraction / test_fraction:
        Proportions of the freely-assignable parties (those not forced to a
        split). The remainder is train. Forced-test accounts are pinned on top,
        so the effective test fraction may exceed test_fraction.
    seed:
        RNG seed; the assignment is deterministic given inputs and seed.
    """

    val_fraction: float = 0.20
    test_fraction: float = 0.15
    seed: int = 1337

    def validate(self) -> None:
        if self.val_fraction < 0.0 or self.test_fraction < 0.0:
            raise ValueError("split fractions must be non-negative.")
        if self.val_fraction + self.test_fraction >= 1.0:
            raise ValueError("val_fraction + test_fraction must be < 1.")


@dataclass(frozen=True, slots=True)
class SplitRecord:
    """Minimal per-account input for splitting: identity and grouping key."""

    account_id: str
    party_id: str | None


@dataclass(frozen=True, slots=True)
class SplitResult:
    """Per-account split assignment, aligned to the input order."""

    account_ids: tuple[str, ...]
    split: tuple[int, ...]

    def summary(self) -> dict[str, int]:
        return {
            "accounts": len(self.account_ids),
            "train": sum(1 for s in self.split if s == Split.TRAIN),
            "val": sum(1 for s in self.split if s == Split.VAL),
            "test": sum(1 for s in self.split if s == Split.TEST),
        }


def _group_accounts_by_party(
    records: Sequence[SplitRecord],
) -> dict[str, list[int]]:
    groups: dict[str, list[int]] = {}
    for i, r in enumerate(records):
        key = r.party_id if r.party_id is not None else f"__solo__{r.account_id}"
        groups.setdefault(key, []).append(i)
    return groups


def _assign_by_fraction(
    keys: list[str], val_fraction: float, test_fraction: float
) -> tuple[set[str], set[str]]:
    # Slice an (already-shuffled) key list into val / test sets by fraction;
    # the remainder is train. Shared by the plain-group and positive-group paths.
    k = len(keys)
    n_val = int(round(val_fraction * k))
    n_test = int(round(test_fraction * k))
    return set(keys[:n_val]), set(keys[n_val : n_val + n_test])


def _stratify_positive_groups(
    keys: list[str], val_fraction: float, test_fraction: float
) -> tuple[set[str], set[str]]:
    # Distribute the FEW groups that contain a revealed positive across the
    # splits, with a floor of one group each for val and test whenever there are
    # at least three such groups. Proportional slicing alone would round the
    # tiny val/test shares down to zero (the bug that left val with no positives
    # to rank for Proxy-AUC model selection); the floor guarantees coverage.
    k = len(keys)
    if k == 0:
        return set(), set()
    if k == 1:
        # the single positive group is most valuable for training the rare class
        return set(), set()
    if k == 2:
        # one to val (model selection needs it), one to train
        return {keys[0]}, set()
    n_val = max(1, int(round(val_fraction * k)))
    n_test = max(1, int(round(test_fraction * k)))
    # never claim more than available; train keeps the remainder
    if n_val + n_test >= k:
        n_test = max(1, k - n_val - 1)
    return set(keys[:n_val]), set(keys[n_val : n_val + n_test])


def split_accounts(
    records: Sequence[SplitRecord],
    config: SplitConfig,
    force_test: frozenset[str] | None = None,
    stratify: frozenset[str] | None = None,
) -> SplitResult:
    """Assign each account to TRAIN, VAL, or TEST, grouped by party.

    Accounts are grouped by party so that all accounts owned by one party land
    in the same split (no owner straddles the train/test boundary, which would
    leak). force_test names accounts that must be in TEST regardless; any party
    containing a forced account is pinned entirely to TEST.

    stratify names accounts (the revealed positives, pu_label==1) whose party
    groups must be spread across train/val/test rather than left to chance. With
    only a handful of revealed positives, proportional slicing can round val's
    share to zero, leaving Proxy-AUC model selection with no positive to rank.
    Positive-bearing groups are therefore assigned first, with a floor of one
    group each for val and test (when at least three exist). The remaining
    (plain) groups are split by val_fraction / test_fraction, remainder to train.
    """
    config.validate()
    forced: frozenset[str] = force_test if force_test is not None else frozenset()
    strat: frozenset[str] = stratify if stratify is not None else frozenset()
    rng = random.Random(config.seed)

    n = len(records)
    split: list[int] = [-1] * n

    groups = _group_accounts_by_party(records)
    group_keys = sorted(groups.keys())
    rng.shuffle(group_keys)

    # Partition the non-forced ("free") groups into those that contain a
    # revealed positive (stratified first, for guaranteed coverage) and the rest.
    positive_groups: list[str] = []
    plain_groups: list[str] = []
    for key in group_keys:
        idxs = groups[key]
        if any(records[i].account_id in forced for i in idxs):
            for i in idxs:
                split[i] = int(Split.TEST)
        elif any(records[i].account_id in strat for i in idxs):
            positive_groups.append(key)
        else:
            plain_groups.append(key)

    pos_val, pos_test = _stratify_positive_groups(
        positive_groups, config.val_fraction, config.test_fraction
    )
    plain_val, plain_test = _assign_by_fraction(
        plain_groups, config.val_fraction, config.test_fraction
    )
    val_groups = pos_val | plain_val
    test_groups = pos_test | plain_test

    for key in positive_groups + plain_groups:
        if key in val_groups:
            s = int(Split.VAL)
        elif key in test_groups:
            s = int(Split.TEST)
        else:
            s = int(Split.TRAIN)
        for i in groups[key]:
            split[i] = s

    for i in range(n):
        if split[i] == -1:
            split[i] = int(Split.TRAIN)

    return SplitResult(
        account_ids=tuple(r.account_id for r in records),
        split=tuple(split),
    )

## 9 · Global-id ↔ integer mapper
`indexing/node_id_mapper.py`

PyG works in contiguous integer node ids; TigerGraph works in global string ids. This per-type, idempotent, resettable mapper is the bridge. The sampler registers ids → integers; the feature store reverses integers → ids to fetch. They **must share one mapper** — that's why a backend object owns it.

In [11]:
from dataclasses import dataclass, field

NodeType = str


class NodeIDMapperError(KeyError):
    pass


@dataclass(slots=True)
class NodeIDMapper:
    """
    Per-batch bidirectional map between global string ids and PyG integer ids.

    For one sampled batch, assigns each global string id a contiguous per-type
    integer so it can live in PyG's node tensor, and reverses it
    so the FeatureStore can recover the string id to fetch features. Reset each
    batch, so it never materializes a global id table regardless of graph scale.
    """

    _to_int: dict[NodeType, dict[str, int]] = field(default_factory=dict)
    _to_str: dict[NodeType, list[str]] = field(default_factory=dict)

    def register(self, node_type: NodeType, string_ids: list[str]) -> list[int]:
        """
        Register an ordered list of global string ids for a node type.

        Returns the assigned integer ids, in the same order. Ids already
        registered keep their previously assigned integer; new ids extend the
        mapping. The returned list lines up positionally with string_ids.
        """
        to_int = self._to_int.setdefault(node_type, {})
        to_str = self._to_str.setdefault(node_type, [])
        out: list[int] = []
        for sid in string_ids:
            existing = to_int.get(sid)
            if existing is None:
                new_id = len(to_str)
                to_int[sid] = new_id
                to_str.append(sid)
                out.append(new_id)
            else:
                out.append(existing)
        return out

    def reset(self) -> None:
        """
        Drop all registered ids, returning the mapper to its empty state.

        The sampler calls this at the start of each batch so the table holds
        only the current batch's nodes, not every node sampled across the run.
        """
        self._to_int.clear()
        self._to_str.clear()

    def to_string(self, node_type: NodeType, int_id: int) -> str:
        """Recover the global string id for an assigned integer id."""
        ids = self._to_str.get(node_type)
        if ids is None or int_id < 0 or int_id >= len(ids):
            raise NodeIDMapperError(f"no string id for {node_type!r} int id {int_id}")
        return ids[int_id]

    def to_strings(self, node_type: NodeType, int_ids: list[int]) -> list[str]:
        """Recover global string ids for a list of assigned integer ids."""
        return [self.to_string(node_type, i) for i in int_ids]

    def to_int(self, node_type: NodeType, string_id: str) -> int:
        """Look up the integer id assigned to a global string id."""
        mapping = self._to_int.get(node_type)
        if mapping is None or string_id not in mapping:
            raise NodeIDMapperError(f"no int id for {node_type!r} string id {string_id!r}")
        return mapping[string_id]

    def num_nodes(self, node_type: NodeType) -> int:
        """Number of registered ids for a node type."""
        return len(self._to_str.get(node_type, []))

    def node_types(self) -> list[NodeType]:
        """All node types that have registered ids."""
        return list(self._to_str.keys())

## 10 · Reindex a sampled neighbourhood
`indexing/reindex.py`

Takes the raw GSQL sample (global ids + edges) and produces **local** integer-indexed tensors with the **seeds ordered first** (rows 0..b-1). `edge_type_schema` is the one map from a PyG edge triple to its raw GSQL relation name.

In [13]:
from collections.abc import Sequence
from dataclasses import dataclass
from typing import cast

NodeType = str
EdgeType = tuple[str, str, str]

_ACCOUNT: NodeType = "Account"


class ReindexError(ValueError):
    pass


_EDGE_TYPE_SCHEMA: dict[str, EdgeType] = {
    "HAS_PAID": ("Account", "HAS_PAID", "Account"),
    "Account_Account": ("Account", "Account_Account", "Account"),
    "Account_Party": ("Account", "Account_Party", "Party"),
    "Party_Entity": ("Party", "Party_Entity", "Resolved_Entity"),
    "Entity_Party": ("Resolved_Entity", "Entity_Party", "Party"),
    "Party_Account": ("Party", "Party_Account", "Account"),
}


def edge_type_schema() -> dict[str, EdgeType]:
    """Mapping from raw GSQL e_type string to PyG (src, relation, dst) triple."""
    return dict(_EDGE_TYPE_SCHEMA)


@dataclass(frozen=True, slots=True)
class RawNeighborhood:
    """Parsed GSQL sampler output: per-type global id lists, plus edges as
    (from_global_id, to_global_id, e_type) triples referencing those ids.
    """

    node_ids: dict[NodeType, list[str]]
    edges: list[tuple[str, str, str]]


@dataclass(frozen=True, slots=True)
class LocalGraph:
    """Local-indexed neighborhood, ready to become a PyG HeteroSamplerOutput.

    node: ordered global ids per type (position == local id).
    row/col: local indices into the per-type lists, per edge type.
    """

    node: dict[NodeType, list[str]]
    row: dict[EdgeType, list[int]]
    col: dict[EdgeType, list[int]]

    def num_nodes(self, node_type: NodeType) -> int:
        return len(self.node.get(node_type, []))

    def num_edges(self, edge_type: EdgeType) -> int:
        return len(self.row.get(edge_type, []))


def _build_index(node_ids: Sequence[str]) -> dict[str, int]:
    index: dict[str, int] = {}
    for i, nid in enumerate(node_ids):
        if nid in index:
            raise ReindexError(f"duplicate node id in input: {nid!r}")
        index[nid] = i
    return index


def _order_seeds_first(account_ids: Sequence[str], seed_ids: Sequence[str]) -> list[str]:
    """
    Reorder account_ids with seeds first, so logits[:batch_size] hits the seeds,
    not neighbors.
    """
    present = set(account_ids)
    seen: set[str] = set()
    ordered_seeds: list[str] = []
    for s in seed_ids:
        if s in present and s not in seen:
            seen.add(s)
            ordered_seeds.append(s)
    rest = [a for a in account_ids if a not in seen]
    return ordered_seeds + rest


def reindex_neighborhood(raw: RawNeighborhood, seed_ids: Sequence[str] | None = None) -> LocalGraph:
    """
    Convert global-id neighborhood to local indices, seeds ordered first.

    Each edge's endpoints are looked up in the per-type node lists to produce
    local row/col indices; an endpoint absent from its list is rejected. With
    seed_ids, Accounts are reordered (seeds first) before indices are built, so
    the first len(seed_ids) Account rows are the seeds.
    """
    node: dict[NodeType, list[str]] = {ntype: list(ids) for ntype, ids in raw.node_ids.items()}
    if seed_ids is not None and _ACCOUNT in node:
        node[_ACCOUNT] = _order_seeds_first(node[_ACCOUNT], seed_ids)

    indices: dict[NodeType, dict[str, int]] = {
        ntype: _build_index(ids) for ntype, ids in node.items()
    }

    row: dict[EdgeType, list[int]] = {}
    col: dict[EdgeType, list[int]] = {}

    for from_id, to_id, e_type in raw.edges:
        schema = _EDGE_TYPE_SCHEMA.get(e_type)
        if schema is None:
            raise ReindexError(f"unknown edge type: {e_type!r}")
        src_type, _, dst_type = schema

        src_index = indices.get(src_type)
        dst_index = indices.get(dst_type)
        if src_index is None:
            raise ReindexError(
                f"edge type {e_type!r} needs source node type {src_type!r}, "
                + "which is absent from the node sets"
            )
        if dst_index is None:
            raise ReindexError(
                f"edge type {e_type!r} needs destination node type {dst_type!r}, "
                + "which is absent from the node sets"
            )

        local_src = src_index.get(from_id)
        local_dst = dst_index.get(to_id)
        if local_src is None:
            raise ReindexError(f"edge {e_type!r}: source id {from_id!r} not in {src_type!r} nodes")
        if local_dst is None:
            raise ReindexError(
                f"edge {e_type!r}: destination id {to_id!r} not in {dst_type!r} nodes"
            )

        row.setdefault(schema, []).append(local_src)
        col.setdefault(schema, []).append(local_dst)

    return LocalGraph(node=node, row=row, col=col)


def parse_raw_result(result: Sequence[object]) -> RawNeighborhood:
    """
    Parse the GSQL sampler result into a RawNeighborhood.

    Reads the per-type id blocks (account_ids/party_ids/entity_ids) and the
    edges block of {from_id, to_id, e_type} objects.
    """
    name_to_type: dict[str, NodeType] = {
        "account_ids": "Account",
        "party_ids": "Party",
        "entity_ids": "Resolved_Entity",
    }
    node_ids: dict[NodeType, list[str]] = {}
    edges: list[tuple[str, str, str]] = []

    for block in result:
        if not isinstance(block, dict):
            continue
        b = cast(dict[str, object], block)
        for key, ntype in name_to_type.items():
            if key in b:
                raw_ids = b[key]
                ids: list[str] = []
                if isinstance(raw_ids, list):
                    for item in cast(list[object], raw_ids):
                        ids.append(str(item))
                node_ids[ntype] = ids
        if "edges" in b:
            raw_edges = b["edges"]
            if isinstance(raw_edges, list):
                for e in cast(list[object], raw_edges):
                    if not isinstance(e, dict):
                        continue
                    ed = cast(dict[str, object], e)
                    from_id = ed.get("from_id")
                    to_id = ed.get("to_id")
                    e_type = ed.get("e_type")
                    if from_id is not None and to_id is not None and e_type is not None:
                        edges.append((str(from_id), str(to_id), str(e_type)))

    return RawNeighborhood(node_ids=node_ids, edges=edges)



## 11 · Connection settings
`tigergraph/settings.py`

Reads host / graph name / secret from the environment (`.env`). No secrets in code.

In [14]:
from typing import ClassVar

from pydantic import Field, SecretStr, ValidationInfo, field_validator
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    model_config: ClassVar[SettingsConfigDict] = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore",
        case_sensitive=False,
    )

    host: str = Field(
        default="",
        description="TigerGraph host URL",
    )
    graphname: str = Field(
        default="",
        description="Name of the graph",
    )
    secret: SecretStr = Field(
        default=SecretStr(""),
        description="REST++ secret used to mint auth tokens",
    )

    @field_validator("host", "graphname")
    @classmethod
    def _str_required(cls, v: str, info: ValidationInfo) -> str:
        if not v:
            name = info.field_name or "field"
            raise ValueError(f"{name} must be set in .env")
        return v

    @field_validator("secret")
    @classmethod
    def _secret_required(cls, v: SecretStr) -> SecretStr:
        if not v.get_secret_value():
            raise ValueError("secret must be set in .env")
        return v

## 12 · TigerGraph client
`tigergraph/client.py`

Thin wrapper over the pyTigerGraph connection — the single handle every query and fetch uses.

In [15]:
from functools import partial
import socket
from typing import cast, override

import requests

from pyTigerGraph import TigerGraphConnection


_READ_TIMEOUT_S = 600.0
_CONNECT_TIMEOUT_S = 30.0


class Client:
    """
    Client that connects to TigerGraph
    """

    _settings: Settings
    conn: TigerGraphConnection

    def __init__(self, settings: Settings) -> None:
        self._settings = settings
        socket.setdefaulttimeout(_READ_TIMEOUT_S)
        self.conn = TigerGraphConnection(
            host=settings.host,
            graphname=settings.graphname,
            gsqlSecret=settings.secret.get_secret_value(),
        )
        _ = self.conn.getToken(settings.secret.get_secret_value())
        self._install_default_timeout()

    def _install_default_timeout(self) -> None:
        session = cast(object, getattr(self.conn, "_session", None))
        if not isinstance(session, requests.Session):
            return
        if getattr(session.request, "_has_default_timeout", False):
            return
        wrapped = partial(session.request, timeout=(_CONNECT_TIMEOUT_S, _READ_TIMEOUT_S))
        setattr(wrapped, "_has_default_timeout", True)
        setattr(session, "request", wrapped)

    @property
    def graphname(self) -> str:
        return self._settings.graphname

    @override
    def __repr__(self) -> str:
        return f"Client(graphname={self._settings.graphname!r})"

## 13 · GSQL file registry
`tigergraph/gsql_paths.py`

Maps a short **registry key** (e.g. `money_flow`) to the `.gsql` file on disk and to its real installed query name. Keeps the notebook's query references stable even when installed names differ.

In [16]:
from pathlib import Path

_GSQL_RELPATHS: dict[str, str] = {
    # schema / loading
    "loading_job": "schema/loading_job.gsql",
    # features
    "account_account_degree": "features/account_account_degree.gsql",
    "derive_max_bins": "features/derive_max_bins.gsql",
    "fastrp": "features/fastrp.gsql",
    "identity_sharing": "features/identity_sharing.gsql",
    "money_flow": "features/money_flow.gsql",
    "pagerank": "features/pagerank.gsql",
    "temporal_features": "features/temporal_features.gsql",
    "triangle_clustering": "features/triangle_clustering.gsql",
    # community detection
    "weight_account_edges": "community_detection/weight_account_edges.gsql",
    "cluster_with_wcc": "community_detection/cluster_with_wcc.gsql",
    # entity resolution
    "match_parties": "entity_resolution/match_parties.gsql",
    "unify_parties": "entity_resolution/unify_parties.gsql",
    # sampling (ML runtime)
    "sample_khop_neighborhood": "sampling/sample_khop_neighborhood.gsql",
    "fetch_account_features": "sampling/fetch_account_features.gsql",
    "fetch_has_paid_features": "sampling/fetch_has_paid_features.gsql",
    # export
    "export_account_features": "export/export_account_features.gsql",
    "export_edges_by_type": "export/export_edges_by_type.gsql",
    "export_has_paid_edges": "export/export_has_paid_edges.gsql",
    # masking inputs
    "get_masking_inputs": "masking/get_masking_inputs.gsql",
    # experiments (throwaway probes / diagnostics)
    "diagnose_export": "experiments/diagnose_export.gsql",
    "diagnose_sentinels": "experiments/diagnose_sentinels.gsql",
    "probe_sample_clause": "experiments/probe_sample_clause.gsql",
    "get_split_accounts": "masking/get_split_accounts.gsql",
    "derive_reference_epoch": "features/derive_reference_epoch.gsql",
}


class GsqlPathError(KeyError):
    pass


def gsql_root() -> Path:
    """
    Absolute path to the repository's top-level gsql/ directory.
    """
    # this file: <repo>/src/mule_pattern_learner/tigergraph/gsql_paths.py
    # parents:    0 tigergraph  1 mule_pattern_learner  2 src  3 <repo>
    repo_root = Path(__file__).resolve().parents[3]
    return repo_root / "gsql"


def gsql_path(query_name: str) -> Path:
    """
    Absolute path to the .gsql file defining the query.
    """
    relpath = _GSQL_RELPATHS.get(query_name)
    if relpath is None:
        known = ", ".join(sorted(_GSQL_RELPATHS))
        raise GsqlPathError(f"unknown query {query_name!r}; known: {known}")
    return gsql_root() / relpath


def query_names() -> list[str]:
    """All known installed-query names."""
    return sorted(_GSQL_RELPATHS)

## 14 · Install / run GSQL queries
`tigergraph/gsql_install.py`

Installs a query from its `.gsql` file and runs it (async/detached, so long feature scans don't drop the connection).

In [17]:
class GsqlInstallError(RuntimeError):
    pass


def get_query_from_file(registry_name: str) -> str:
    """
     Query name for a gsql_paths registry key.

    Parses `CREATE [OR REPLACE] [DISTRIBUTED] QUERY <name>` from the .gsql file.
    """
    text = gsql_path(registry_name).read_text(encoding="utf-8")
    match = _CREATE_QUERY_RE.search(text)
    if match is None:
        raise GsqlInstallError(
            f"{registry_name}: no CREATE QUERY found in source; not an installable query "
            "(a loading job or schema script installs differently)."
        )
    return match.group(1)


def _run_gsql(client: Client, statement: str) -> str:
    result = client.conn.gsql(statement)
    if not isinstance(result, str):
        raise GsqlInstallError(f"expected text from gsql(), got {type(result).__name__}")
    return result


def _check_output(registry_name: str, action: str, output: str) -> None:
    """
    Raise GsqlInstallError unless the install log confirms success.

    Success-first: if TigerGraph printed a success marker, the install compiled
    and we return -- even if the log also contains a benign "0 error(s)" line.
    Only when NO success marker is present do we scan for failure phrases. This
    is why bare "error"/"fail" are no longer needed (and were removed): they
    matched success logs and produced false failures (e.g. match_parties).
    """
    low = output.lower()
    if any(marker in low for marker in _SUCCESS_MARKERS):
        return
    if any(marker in low for marker in _FAILURE_MARKERS):
        raise GsqlInstallError(
            f"{action} {registry_name!r} reported a problem:\n{output.strip()[:800]}"
        )
    # No success marker AND no known failure phrase: ambiguous. Surface it
    # rather than silently assuming success -- a swallowed failure here would
    # let the pipeline build on a query that never installed.
    raise GsqlInstallError(
        f"{action} {registry_name!r} produced no success confirmation:\n{output.strip()[:800]}"
    )


def install_query(client: Client, registry_name: str, drop_first: bool = True) -> str:
    """
    Install one query from its .gsql file; return the install log.

    """
    path = gsql_path(registry_name)
    if not path.is_file():
        raise GsqlInstallError(f"{registry_name}: source file not found at {path}")
    name = get_query_from_file(registry_name)
    text = path.read_text(encoding="utf-8")

    if drop_first:

        drop_out = _run_gsql(client, f"USE GRAPH {client.graphname}\nDROP QUERY {name}\n")
        low = drop_out.lower()
        dropped_ok = any(marker in low for marker in _DROP_SUCCESS_MARKERS)
        if not dropped_ok and any(m in low for m in _FAILURE_MARKERS):
            raise GsqlInstallError(f"drop {name!r} failed:\n{drop_out.strip()[:800]}")

    install_out = _run_gsql(client, text + f"\nINSTALL QUERY {name}\n")
    _check_output(registry_name, "install", install_out)
    return install_out


def install_queries(
    client: Client, registry_names: list[str], drop_first: bool = True
) -> dict[str, str]:
    """
    Installs queries in order; return {registry_name: install log}.
    """
    logs: dict[str, str] = {}
    for registry_name in registry_names:
        logs[registry_name] = install_query(client, registry_name, drop_first=drop_first)
    return logs


def _extract_request_id(submitted: object) -> str:
    # runInstalledQuery(runAsync=True) returns the detached-mode request id. It
    # is normally a bare string, but tolerate a 1-element list or a dict with a
    # request id so a pyTigerGraph version change does not silently break this.
    if isinstance(submitted, str):
        return submitted
    if isinstance(submitted, list) and len(submitted) == 1 and isinstance(submitted[0], str):
        return submitted[0]
    if isinstance(submitted, dict):
        for key in ("request_id", "requestid", "requestId"):
            value = submitted.get(key)
            if isinstance(value, str):
                return value
    raise GsqlInstallError(f"could not read a request id from async submission: {submitted!r}")


def _status_of(status_response: object) -> str:
    # checkQueryStatus returns a list of {status: success|running|aborted, ...}.
    # We submit one query at a time, so read the first entry's status. The
    # failure markers are deliberately NOT applied here: a "running" status
    # legitimately contains no success/failure phrase, and reusing _check_output
    # would misread an in-progress poll as a failure.
    rows: list[object]
    if isinstance(status_response, list):
        rows = cast(list[object], status_response)
    elif isinstance(status_response, dict):
        results = status_response.get("results")
        rows = cast(list[object], results) if isinstance(results, list) else [status_response]
    else:
        raise GsqlInstallError(f"unexpected query-status payload: {status_response!r}")
    if not rows:
        raise GsqlInstallError("empty query-status payload")
    first = rows[0]
    if not isinstance(first, dict):
        raise GsqlInstallError(f"unexpected query-status row: {first!r}")
    status = cast(dict[str, object], first).get("status")
    if not isinstance(status, str):
        raise GsqlInstallError(f"query-status row has no string status: {first!r}")
    return status


def run_query(
    client: Client,
    registry_name: str,
    params: dict[str, object] | None = None,
    poll_seconds: float = 5.0,
    max_wait_seconds: float = 14_400.0,
    query_timeout_ms: int = 7_200_000,
) -> list[object]:
    """
    Run an installed query by its gsql_paths registry key, in detached (async)
    mode, and block until it finishes.

    On Savanna a synchronous runInstalledQuery holds one HTTPS connection open
    for the query's whole duration; long feature queries on a large graph
    outlive the managed load balancer's idle window and the connection is
    dropped (surfacing as ReadTimeout even with read timeout=None -- the value
    is irrelevant because the socket is severed, not timed out). Detached mode
    avoids this: the query runs server-side and we issue only short calls --
    submit, then poll, then fetch -- so there is no long-lived connection to
    drop. The feature queries are idempotent, so a re-run after any transient
    failure is safe.

    poll_seconds: gap between status checks. max_wait_seconds: hard ceiling on
    total wait (default 4h) so a genuinely stuck query cannot hang forever.
    query_timeout_ms: the SERVER-SIDE limit (milliseconds) TigerGraph applies
    before it aborts the query itself, the same value a GSQL session sets with
    SET query_timeout. This is distinct from max_wait_seconds (client-side poll
    ceiling): heavy feature queries (fastrp, pagerank) far exceed RESTPP's small
    default and are aborted server-side unless this is raised. Default 2h.
    """
    name = get_query_from_file(registry_name)
    submitted: object = client.conn.runInstalledQuery(
        name,
        params if params is not None else {},
        timeout=query_timeout_ms,
        runAsync=True,
    )
    request_id = _extract_request_id(submitted)

    waited = 0.0
    while True:
        status = _status_of(client.conn.checkQueryStatus(request_id))
        if status == "success":
            break
        if status in ("aborted", "timeout"):
            raise GsqlInstallError(
                f"async query {name!r} ended with status {status!r} "
                + f"(request_id {request_id})."
            )
        if waited >= max_wait_seconds:
            raise GsqlInstallError(
                f"async query {name!r} still {status!r} after {max_wait_seconds:.0f}s "
                + f"(request_id {request_id}); aborting the wait."
            )
        time.sleep(poll_seconds)
        waited += poll_seconds

    return cast(list[object], client.conn.getQueryResult(request_id))

## 15 · Derive temporal spec from the graph
`tigergraph/derivation.py`

Asks the graph itself for `max_bins` and the reference epoch (latest txn), so `edge_dim=34` and the time origin are **derived from the data**, never hardcoded.

In [18]:
from dataclasses import dataclass
from typing import cast


_DERIVE_QUERY_NAME = "derive_max_bins"
_REFERENCE_EPOCH_QUERY_NAME = "derive_reference_epoch"


class TemporalDerivationError(RuntimeError):
    pass


@dataclass(frozen=True, slots=True)
class GraphTemporalSpec:
    """
    This is the single source of truth for the HAS_PAID bin width within a run.
    It is produced once by derive_temporal_spec() and then passed to both the
    edge-feature builder (padding width) and the model (edge_dim), so the two
    can never disagree. It holds no database handle and performs no I/O.
    """

    max_bins: int

    def __post_init__(self) -> None:
        if self.max_bins < 1:
            raise TemporalDerivationError(f"max_bins must be >= 1, got {self.max_bins}")

    @property
    def edge_dim(self) -> int:
        """Flattened per-edge feature width a fixed-edge-dim GNN layer expects."""
        return flat_edge_dim(self.max_bins)


def _scalar_int(raw: list[object], key: str) -> int:
    for block in raw:
        if isinstance(block, dict) and key in block:
            value = cast(dict[str, object], block)[key]
            if isinstance(value, bool) or not isinstance(value, int):
                raise TemporalDerivationError(
                    f"{_DERIVE_QUERY_NAME}: {key!r} was not an int: {value!r}"
                )
            return value
    raise TemporalDerivationError(f"{_DERIVE_QUERY_NAME}: key {key!r} not found in output: {raw!r}")


def derive_temporal_spec(client: Client) -> GraphTemporalSpec:
    """Run derive_max_bins against the graph and build a GraphTemporalSpec.

    Performs a full-edge scan on the server (the derive_max_bins query) and
    verifies that the declared num_bins attribute agrees with the actual
    amount_bins / count_bins list lengths; a mismatch means padding would
    silently truncate real data, so it is raised rather than tolerated.
    """
    raw = cast(list[object], client.conn.runInstalledQuery(_DERIVE_QUERY_NAME, {}))
    max_bins = _scalar_int(raw, "max_bins")
    amount_len = _scalar_int(raw, "max_amount_len")
    count_len = _scalar_int(raw, "max_count_len")

    if not (max_bins == amount_len == count_len):
        raise TemporalDerivationError(
            "inconsistent bin counts in graph data: "
            + f"num_bins={max_bins}, amount_bins.size()={amount_len}, "
            + f"count_bins.size()={count_len}. The declared num_bins disagrees "
            + "with actual list lengths; padding would truncate real temporal data."
        )

    return GraphTemporalSpec(max_bins=max_bins)


def derive_reference_epoch_s(client: Client) -> float:
    """Run derive_reference_epoch against the graph; return the snapshot epoch.

    The reference epoch is the latest transaction time in the graph (unix
    seconds), the snapshot moment recency edge-features are measured against.
    Deriving it keeps the loader consistent with the data and avoids a "now"
    that post-dates it. Performs a full HAS_PAID scan on the server.
    """
    raw = cast(list[object], client.conn.runInstalledQuery(_REFERENCE_EPOCH_QUERY_NAME, {}))
    return float(_scalar_int(raw, "reference_epoch_s"))

## 16 · Neighbour fanout
`pyg/neighbors.py`

Per-edge-type fanout: how many neighbours to sample at each hop, for each relation.

In [19]:
from dataclasses import dataclass


class NeighborFanoutError(ValueError):
    pass


@dataclass(frozen=True, slots=True)
class NeighborFanout:
    """Per-relation neighbor counts for the two-round k-hop sampler.

    A parameter object (in PyG's NumNeighbors spirit): a small, typed, validated
    bundle of how many neighbors are drawn per relation per round. It is the
    single source of truth for sampler fanout. Round-2 counts are intentionally
    smaller than round-1 to control neighborhood explosion (standard
    decreasing-fanout practice).

    """

    has_paid: int = 15
    account_account: int = 10
    acct_party: int = 5
    party_entity: int = 5
    entity_party: int = 10
    party_acct: int = 10
    has_paid_round2: int = 5
    account_account_round2: int = 5
    acct_party_round2: int = 3
    party_entity_round2: int = 3
    entity_party_round2: int = 5
    party_acct_round2: int = 5

    def __post_init__(self) -> None:
        for name, value in self._counts().items():
            if isinstance(value, bool):
                raise NeighborFanoutError(f"{name}: fanout must be an int, got bool")
            if value < 1:
                raise NeighborFanoutError(f"{name}: fanout must be >= 1, got {value}")

    def _counts(self) -> dict[str, int]:
        """All fanout fields as an explicitly typed name -> value mapping."""
        return {
            "has_paid": self.has_paid,
            "account_account": self.account_account,
            "acct_party": self.acct_party,
            "party_entity": self.party_entity,
            "entity_party": self.entity_party,
            "party_acct": self.party_acct,
            "has_paid_round2": self.has_paid_round2,
            "account_account_round2": self.account_account_round2,
            "acct_party_round2": self.acct_party_round2,
            "party_entity_round2": self.party_entity_round2,
            "entity_party_round2": self.entity_party_round2,
            "party_acct_round2": self.party_acct_round2,
        }

    def as_query_params(self) -> dict[str, int]:
        """Return the fanouts keyed by the GSQL query's parameter names.

        The sample_khop_neighborhood query names its round-2 parameters with a
        trailing '_2'; that GSQL-specific spelling is confined to this method.
        """
        return {
            "fanout_has_paid": self.has_paid,
            "fanout_account_account": self.account_account,
            "fanout_acct_party": self.acct_party,
            "fanout_party_entity": self.party_entity,
            "fanout_entity_party": self.entity_party,
            "fanout_party_acct": self.party_acct,
            "fanout_has_paid_2": self.has_paid_round2,
            "fanout_account_account_2": self.account_account_round2,
            "fanout_acct_party_2": self.acct_party_round2,
            "fanout_party_entity_2": self.party_entity_round2,
            "fanout_entity_party_2": self.entity_party_round2,
            "fanout_party_acct_2": self.party_acct_round2,
        }

## 17 · Fetch vertices / edges
`pyg/fetch.py`

The two on-demand fetches: account vertex rows (for node features) and internal HAS_PAID edges (for edge features). Both short-circuit on an empty id list (no server call).

In [20]:
from typing import cast



class FeatureFetchError(RuntimeError):
    pass


_ACCOUNT_FEATURE_QUERY = "fetch_account_features"
_HAS_PAID_FEATURE_QUERY = "fetch_has_paid_features"


def fetch_account_vertices(
    client: Client,
    account_ids: list[str],
    query_name: str = _ACCOUNT_FEATURE_QUERY,
) -> list[object]:
    """Fetch raw Account vertices ({"v_id", "attributes"}) for the given ids.

    Returns the rows in pyTigerGraph's shape, ready for build_account_features.
    Empty id list short-circuits without a server call.
    """
    if not account_ids:
        return []

    params: dict[str, object] = {"ids": [(aid,) for aid in account_ids]}
    raw = client.conn.runInstalledQuery(query_name, params)

    for block in raw:
        if not isinstance(block, dict):
            continue
        b = cast(dict[str, object], block)
        if "Accounts" in b:
            vertices = b["Accounts"]
            if not isinstance(vertices, list):
                raise FeatureFetchError(f"'Accounts' is not a list: {vertices!r}")
            return cast(list[object], vertices)

    raise FeatureFetchError(f"query {query_name!r} returned no 'Accounts' block: {raw!r}")


def fetch_has_paid_edges(
    client: Client,
    account_ids: list[str],
    query_name: str = _HAS_PAID_FEATURE_QUERY,
) -> list[object]:
    """Fetch HAS_PAID edges with both endpoints in account_ids.

    The query nests edges under each source vertex; this flattens them into one
    list of {"from_id", "to_id", "attributes"}, ready for build_edge_features.
    Empty id list short-circuits without a server call.
    """
    if not account_ids:
        return []

    params: dict[str, object] = {"ids": [(aid,) for aid in account_ids]}
    raw = client.conn.runInstalledQuery(query_name, params)

    edges: list[object] = []
    found_block = False
    for block in raw:
        if not isinstance(block, dict):
            continue
        b = cast(dict[str, object], block)
        if "Collected" not in b:
            continue
        found_block = True
        vertices = b["Collected"]
        if not isinstance(vertices, list):
            raise FeatureFetchError(f"'Collected' is not a list: {vertices!r}")
        for vertex in cast(list[object], vertices):
            if not isinstance(vertex, dict):
                raise FeatureFetchError(f"vertex is not a dict: {vertex!r}")
            v = cast(dict[str, object], vertex)
            attrs = v.get("attributes")
            if not isinstance(attrs, dict):
                raise FeatureFetchError(f"vertex missing attributes: {v!r}")
            a = cast(dict[str, object], attrs)
            out_edges = a.get("out_edges")
            if not isinstance(out_edges, list):
                raise FeatureFetchError(f"'out_edges' is not a list: {out_edges!r}")
            edges.extend(cast(list[object], out_edges))

    if not found_block:
        raise FeatureFetchError(f"query {query_name!r} returned no 'Collected' block: {raw!r}")
    return edges

## 18 · PyG FeatureStore (node features)
`pyg/feature_store.py`

Implements PyG's FeatureStore over TigerGraph: given the integer ids PyG asks for, reverse them to string ids (shared mapper), fetch the rows, build + normalize features, and return the tensor **aligned to the requested order**. PyG's remote backend serves **node** features this way.

In [21]:
from typing import override

import torch
from torch import Tensor
from torch_geometric.data import FeatureStore, TensorAttr
from torch_geometric.typing import FeatureTensorType, NodeType



class FeatureStoreError(RuntimeError):
    pass


_ACCOUNT: NodeType = "Account"
_NODE_FEATURE_ATTR: str = "x"

_FEATURE_DIMS: dict[NodeType, int] = {_ACCOUNT: NUM_ACCOUNT_FEATURES}


class TigerGraphFeatureStore(FeatureStore):
    """
    PyG FeatureStore that serves Account node features from TigerGraph.

    PyG calls _get_tensor with integer node ids. This reverses them to global
    string ids via the shared NodeIDMapper, fetches and builds those accounts'
    features, normalizes, and returns a tensor aligned to the requested order.
    Read-only.
    """

    _client: Client
    _mapper: NodeIDMapper
    _normalizer: FeatureNormalizer | None

    def __init__(
        self,
        client: Client,
        mapper: NodeIDMapper,
        normalizer: FeatureNormalizer | None = None,
    ) -> None:
        super().__init__()
        self._client = client
        self._mapper = mapper
        self._normalizer = normalizer

    def _index_to_ids(self, group_name: NodeType, index: Tensor) -> list[str]:
        int_ids: list[int] = [int(i) for i in index.tolist()]
        return self._mapper.to_strings(group_name, int_ids)

    @override
    def _get_tensor(self, attr: TensorAttr) -> Tensor | None:
        group_name = attr.group_name
        attr_name = attr.attr_name
        index = attr.index
        if group_name is None or attr_name is None or index is None:
            raise FeatureStoreError(f"TensorAttr is not fully specified: {attr!r}")
        if attr_name != _NODE_FEATURE_ATTR:
            raise FeatureStoreError(
                f"unsupported attr_name {attr_name!r}; only {_NODE_FEATURE_ATTR!r}"
            )
        if group_name != _ACCOUNT:
            type_list = sorted(_FEATURE_DIMS)
            raise FeatureStoreError(
                f"no features for node type {group_name!r}; types are {type_list}"
            )
        if not isinstance(index, Tensor):
            raise FeatureStoreError(
                f"index must be a Tensor of node ids, got {type(index).__name__}"
            )

        string_ids = self._index_to_ids(group_name, index)
        vertices = fetch_account_vertices(self._client, string_ids)
        features = build_account_features(vertices)

        aligned = self._align(features.node_ids, features.feats, string_ids)
        if self._normalizer is not None:
            aligned = self._normalizer.apply(aligned)
        return aligned

    def _align(self, fetched_ids: tuple[str, ...], feats: Tensor, requested: list[str]) -> Tensor:
        position: dict[str, int] = {sid: i for i, sid in enumerate(fetched_ids)}
        rows: list[int] = []
        for sid in requested:
            row = position.get(sid)
            if row is None:
                raise FeatureStoreError(f"feature row missing for id {sid!r}")
            rows.append(row)
        order = torch.tensor(rows, dtype=torch.long)
        return feats[order]

    @override
    def _put_tensor(self, tensor: FeatureTensorType, attr: TensorAttr) -> bool:
        _ = tensor
        _ = attr
        return False

    @override
    def _remove_tensor(self, attr: TensorAttr) -> bool:
        _ = attr
        return False

    @override
    def _get_tensor_size(self, attr: TensorAttr) -> tuple[int, ...] | None:
        group_name = attr.group_name
        if group_name is None:
            return None
        dim = _FEATURE_DIMS.get(group_name)
        if dim is None:
            return None
        return (dim,)

    @override
    def get_all_tensor_attrs(self) -> list[TensorAttr]:
        return [
            TensorAttr(group_name=ntype, attr_name=_NODE_FEATURE_ATTR) for ntype in _FEATURE_DIMS
        ]

## 19 · PyG GraphStore (edges)
`pyg/graph_store.py`

Implements PyG's GraphStore: the edge-type metadata PyG reads at loader construction, plus paged export of whole relations on demand 

In [22]:
from typing import cast, override

import torch
from torch_geometric.data import EdgeAttr, EdgeLayout, GraphStore
from torch_geometric.typing import EdgeTensorType, EdgeType



class GraphStoreError(RuntimeError):
    pass


# PyG (src, relation, dst) triple -> raw GSQL relation name (reverse of the
# reindex schema). This is the single place the PyG<->GSQL edge naming lives.
_RELATION_BY_TYPE: dict[EdgeType, str] = {
    triple: name for name, triple in edge_type_schema().items()
}


class TigerGraphGraphStore(GraphStore):
    """
    PyG GraphStore over TigerGraph, sharing a backend's mapper.

    Used on the training path via get_all_edge_attrs: PyG's NodeLoader calls it
    at construction to learn the heterogeneous edge types. (Batch connectivity
    itself comes from the k-hop sampler, which queries the server directly, not
    from this store.)

    Edges are static, so put/remove are unsupported.
    """

    _client: Client
    _mapper: NodeIDMapper
    _export_query: str

    def __init__(
        self,
        client: Client,
        mapper: NodeIDMapper,
        export_query: str = "export_edges_by_type",
    ) -> None:
        super().__init__()
        self._client = client
        self._mapper = mapper
        self._export_query = export_query

    @override
    def get_all_edge_attrs(self) -> list[EdgeAttr]:
        return [
            EdgeAttr(edge_type=triple, layout=EdgeLayout.COO, is_sorted=False)
            for triple in edge_type_schema().values()
        ]

    @override
    def _get_edge_index(self, edge_attr: EdgeAttr) -> EdgeTensorType | None:
        edge_type = edge_attr.edge_type
        relation = _RELATION_BY_TYPE.get(edge_type)
        if relation is None:
            raise GraphStoreError(f"unknown edge type {edge_type!r}")

        src_type = edge_type[0]
        dst_type = edge_type[2]
        src_ids, dst_ids = self._export_all(relation, src_type, dst_type)

        row = self._mapper.register(src_type, src_ids)
        col = self._mapper.register(dst_type, dst_ids)
        row_t = torch.tensor(row, dtype=torch.long)
        col_t = torch.tensor(col, dtype=torch.long)
        return (row_t, col_t)

    def _export_all(
        self, relation: str, src_type: str, dst_type: str
    ) -> tuple[list[str], list[str]]:
        # Page through the keyset cursor until the source set is exhausted, so
        # an arbitrarily large relation is pulled in bounded pages rather than
        # one unbounded response. src_type/dst_type are used by the caller to map
        # ids; the query itself derives endpoints from the relation name.
        _ = src_type
        _ = dst_type
        src_ids: list[str] = []
        dst_ids: list[str] = []
        cursor = ""
        while True:
            params: dict[str, object] = {
                "relation": relation,
                "cursor": cursor,
            }
            raw = self._client.conn.runInstalledQuery(self._export_query, params)
            page_src, page_dst, next_cursor = self._parse_page(raw)
            src_ids.extend(page_src)
            dst_ids.extend(page_dst)
            if not next_cursor or next_cursor == cursor:
                break
            cursor = next_cursor
        return src_ids, dst_ids

    def _parse_page(self, raw: list[object]) -> tuple[list[str], list[str], str]:
        src_ids: list[str] = []
        dst_ids: list[str] = []
        next_cursor = ""
        for block in raw:
            if not isinstance(block, dict):
                continue
            b = _string_keyed(block)
            if "edges" in b:
                edges = b["edges"]
                if not isinstance(edges, list):
                    raise GraphStoreError(f"'edges' is not a list: {edges!r}")
                for edge in cast("list[object]", edges):
                    if not isinstance(edge, dict):
                        raise GraphStoreError(f"edge is not a dict: {edge!r}")
                    e = _string_keyed(edge)
                    frm = e.get("from_id")
                    to = e.get("to_id")
                    if not isinstance(frm, str) or not isinstance(to, str):
                        raise GraphStoreError(f"edge missing endpoints: {e!r}")
                    src_ids.append(frm)
                    dst_ids.append(to)
            if "next_cursor" in b:
                nc = b["next_cursor"]
                if isinstance(nc, str):
                    next_cursor = nc
        return src_ids, dst_ids, next_cursor

    @override
    def _put_edge_index(self, edge_index: EdgeTensorType, edge_attr: EdgeAttr) -> bool:
        _ = edge_index
        _ = edge_attr
        return False

    @override
    def _remove_edge_index(self, edge_attr: EdgeAttr) -> bool:
        _ = edge_attr
        return False


def _string_keyed(value: dict[object, object]) -> dict[str, object]:
    return {str(k): v for k, v in value.items()}

## 20 · Hetero k-hop sampler
`pyg/sampler.py`

Wraps the installed `sample_khop_neighborhood` GSQL query as a PyG sampler. **One query call per batch** (a whole batch of seeds expands together), with val/test filtering for leakage control. Registers sampled nodes into the shared mapper, seeds first.

In [23]:
from typing import override

import torch
from torch import Tensor
from torch_geometric.sampler import (
    BaseSampler,
    EdgeSamplerInput,
    HeteroSamplerOutput,
    NegativeSampling,
    NodeSamplerInput,
)
from torch_geometric.typing import EdgeType, NodeType



class TigerGraphSamplerError(RuntimeError):
    pass


class TigerGraphHeteroSampler(BaseSampler):
    """
    A PyG BaseSampler that delegates k-hop neighborhood sampling to TigerGraph.

    Samples each batch's neighborhood with a server-side query and returns it as
    PyG structure, registering the batch's nodes in the shared NodeIDMapper so the
    FeatureStore can fetch their features. seed_ids defines the index<->id mapping,
    bounded by the seed set, not the full graph.
    allow_val/allow_test gate strict-
    inductive filtering: when False, neighborhoods don't traverse into val/test
    accounts (no leakage).
    Train passes both False; val (True, False); test both True.
    """

    _client: Client
    _fanout: NeighborFanout
    _seed_ids: tuple[str, ...]
    _mapper: NodeIDMapper
    _query_name: str
    _allow_val: bool
    _allow_test: bool

    def __init__(
        self,
        client: Client,
        seed_ids: tuple[str, ...],
        mapper: NodeIDMapper,
        fanout: NeighborFanout | None = None,
        query_name: str = "sample_khop_neighborhood",
        allow_val: bool = True,
        allow_test: bool = True,
    ) -> None:
        super().__init__()
        self._client = client
        self._seed_ids = seed_ids
        self._mapper = mapper
        self._fanout = fanout if fanout is not None else NeighborFanout()
        self._query_name = query_name
        self._allow_val = allow_val
        self._allow_test = allow_test

    def _seed_indices_to_ids(self, node: Tensor) -> list[str]:
        indices: list[int] = node.tolist()
        n = len(self._seed_ids)
        ids: list[str] = []
        for idx in indices:
            if idx < 0 or idx >= n:
                raise TigerGraphSamplerError(f"seed index {idx} out of range for {n} seed ids")
            ids.append(self._seed_ids[idx])
        return ids

    def _run_query(self, seed_ids: list[str]) -> list[object]:
        params: dict[str, object] = {"seeds": [(sid,) for sid in seed_ids]}
        for key, value in self._fanout.as_query_params().items():
            params[key] = value
        # strict-inductive split flags: gate traversal into held-out accounts
        params["allow_val"] = self._allow_val
        params["allow_test"] = self._allow_test
        return self._client.conn.runInstalledQuery(self._query_name, params)

    def _to_hetero_output(self, local: LocalGraph, index: NodeSamplerInput) -> HeteroSamplerOutput:
        """
        Pack a reindexed LocalGraph into PyG's HeteroSamplerOutput.

        Registers each node's global id in the shared mapper and writes the
        assigned integers into the node tensors, so the FeatureStore can reverse
        them to fetch features. edge is all-None (edge features are attached
        later by the transform, not at sample time).
        """
        node: dict[NodeType, Tensor] = {}
        for ntype, ids in local.node.items():
            int_ids = self._mapper.register(ntype, list(ids))
            node[ntype] = torch.tensor(int_ids, dtype=torch.long)
        row: dict[EdgeType, Tensor] = {
            etype: torch.tensor(rows, dtype=torch.long) for etype, rows in local.row.items()
        }
        col: dict[EdgeType, Tensor] = {
            etype: torch.tensor(cols, dtype=torch.long) for etype, cols in local.col.items()
        }
        edge: dict[EdgeType, Tensor | None] = {etype: None for etype in local.row}
        batch_size = int(index.node.shape[0])
        metadata: tuple[Tensor | None, int] = (index.input_id, batch_size)
        return HeteroSamplerOutput(node=node, row=row, col=col, edge=edge, metadata=metadata)

    @override
    def sample_from_nodes(self, index: NodeSamplerInput, **kwargs: object) -> HeteroSamplerOutput:
        _ = kwargs
        # Reset the shared mapper so it holds only this batch's nodes, not every
        # node sampled across the run. Loader is synchronous (num_workers=0), so
        # the prior batch's feature fetch is already done and no live id is lost.
        self._mapper.reset()
        seed_ids = self._seed_indices_to_ids(index.node)
        raw = self._run_query(seed_ids)
        # order seeds first so the first len(seeds) Account rows ARE the seeds
        # (the contract the loader relies on to slice seed logits)
        local = reindex_neighborhood(parse_raw_result(raw), seed_ids=seed_ids)
        return self._to_hetero_output(local, index)

    @override
    def sample_from_edges(
        self,
        index: EdgeSamplerInput,
        neg_sampling: NegativeSampling | None = None,
        **kwargs: object,
    ) -> HeteroSamplerOutput:
        _ = index
        _ = neg_sampling
        _ = kwargs
        raise NotImplementedError(
            "TigerGraphHeteroSampler supports node-level sampling only; use NodeLoader."
        )

## 21 · Edge-feature attacher (the HAS_PAID payoff)
`pyg/transform.py`

PyG's remote backend serves node features but **not** edge features. This loader **transform** fetches the HAS_PAID features for exactly the edges the sampler chose and aligns them row-for-row onto `edge_attr`. This is how the payment edge's own features reach the model.

In [24]:
from typing import Protocol, cast

import torch
from torch import Tensor
from torch_geometric.data import HeteroData
from torch_geometric.typing import EdgeType, NodeType


_HAS_PAID: EdgeType = ("Account", "HAS_PAID", "Account")


class _EdgeStore(Protocol):
    edge_index: Tensor
    edge_attr: Tensor


class _NodeStore(Protocol):
    n_id: Tensor
    x: Tensor


def _edge_store(data: HeteroData, key: EdgeType) -> _EdgeStore:
    return cast(_EdgeStore, cast(object, data[key]))


def _node_store(data: HeteroData, key: NodeType) -> _NodeStore:
    return cast(_NodeStore, cast(object, data[key]))


class EdgeFeatureError(RuntimeError):
    pass


class HasPaidEdgeFeatureAttacher:
    """
    Batch transform that attaches HAS_PAID temporal edge features.

    PyG's remote FeatureStore does not serve edge features (node features only),
    so per PyG's design edge features are attached with a loader transform. For
    each sampled batch this fetches the HAS_PAID edges among the batch's Account
    nodes, builds their temporal features (temporal.py), and writes them to
    edge_attr aligned row-for-row to the batch's existing HAS_PAID edge_index.

    It shares the backend's mapper to turn the batch's local Account integers
    back into global ids for the fetch and the alignment.
    """

    _client: Client
    _mapper: NodeIDMapper
    _reference_epoch_s: float
    _max_bins: int

    def __init__(
        self,
        client: Client,
        mapper: NodeIDMapper,
        reference_epoch_s: float,
        max_bins: int,
    ) -> None:
        self._client = client
        self._mapper = mapper
        self._reference_epoch_s = reference_epoch_s
        self._max_bins = max_bins

    def __call__(self, data: HeteroData) -> HeteroData:
        if _HAS_PAID not in data.edge_types:
            return data

        edge_store = _edge_store(data, _HAS_PAID)
        edge_index = edge_store.edge_index
        num_edges = int(edge_index.shape[1])
        dim = flat_edge_dim(self._max_bins)

        if num_edges == 0:
            edge_store.edge_attr = torch.zeros((0, dim), dtype=torch.float32)
            return data

        account_int = _node_store(data, "Account").n_id
        account_int_ids: list[int] = [int(i) for i in account_int.tolist()]
        account_ids = self._mapper.to_strings("Account", account_int_ids)

        raw_edges = fetch_has_paid_edges(self._client, account_ids)
        features = build_edge_features(raw_edges, self._reference_epoch_s, self._max_bins)
        flat = flat_edge_features(features)

        lookup: dict[tuple[str, str], int] = {}
        for i, (s, d) in enumerate(zip(features.src_ids, features.dst_ids)):
            lookup[(s, d)] = i

        row: list[int] = [int(r) for r in edge_index[0].tolist()]
        col: list[int] = [int(c) for c in edge_index[1].tolist()]
        order: list[int] = []
        for r, c in zip(row, col):
            src_id = account_ids[r]
            dst_id = account_ids[c]
            pos = lookup.get((src_id, dst_id))
            if pos is None:
                raise EdgeFeatureError(f"no fetched features for edge {src_id} -> {dst_id}")
            order.append(pos)

        edge_store.edge_attr = flat[torch.tensor(order, dtype=torch.long)]
        return data

## 22 · The model — heterogeneous GATv2
`pyg/model.py`

`MulePatternModel`. A `Linear(31→64)` input projection; **two** `HeteroConv` layers (one `GATv2Conv` per edge type, summed over relations) because the sampler fetches 2 hops; `edge_dim=34` passed **only** to the HAS_PAID conv; featureless node types get learned embeddings; a readout head → **one logit per account**. The attention maths lives inside PyG's `GATv2Conv`.

In [25]:
from typing import cast, override

import torch
from torch import Tensor
from torch.nn import Dropout, Embedding, Linear, Module, ModuleDict
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, HeteroConv, MessagePassing
from torch_geometric.typing import EdgeType, NodeType

# The six heterogeneous edge types the sampler produces. Same-type edges
# (Account->Account) admit self-loops; the four cross-type (bipartite) relations
# do not, since a self-loop across distinct node types is undefined.
_HAS_PAID: EdgeType = ("Account", "HAS_PAID", "Account")
_ACCOUNT_ACCOUNT: EdgeType = ("Account", "Account_Account", "Account")
_ACCOUNT_PARTY: EdgeType = ("Account", "Account_Party", "Party")
_PARTY_ENTITY: EdgeType = ("Party", "Party_Entity", "Resolved_Entity")
_ENTITY_PARTY: EdgeType = ("Resolved_Entity", "Entity_Party", "Party")
_PARTY_ACCOUNT: EdgeType = ("Party", "Party_Account", "Account")

_EDGE_TYPES: tuple[EdgeType, ...] = (
    _HAS_PAID,
    _ACCOUNT_ACCOUNT,
    _ACCOUNT_PARTY,
    _PARTY_ENTITY,
    _ENTITY_PARTY,
    _PARTY_ACCOUNT,
)
_SAME_TYPE_EDGES: frozenset[EdgeType] = frozenset({_HAS_PAID, _ACCOUNT_ACCOUNT})

_ACCOUNT: NodeType = "Account"


class MulePatternModel(Module):
    """
    Heterogeneous GATv2 model that scores accounts by mule-likelihood.

    The attention mathematics below lives INSIDE each GATv2Conv (PyG implements
    it); this class declares the convs with the right dimensions and wires them
    together. For one attention head, a destination node i is updated from its
    neighbors j as:

        score:      e_ij = a^T . LeakyReLU( W . h_i + W . h_j + W_e . edge_ij )
        normalize:  alpha_ij = softmax_j(e_ij) = exp(e_ij) / sum_k exp(e_ik)
        aggregate:  h_i' = sigma( sum_j alpha_ij * (W . h_j) )

    "v2" puts the learnable vector a AFTER the LeakyReLU (dynamic attention), so
    node i can rank its neighbors differently than another node would. With H
    heads the per-head outputs are concatenated:

        h_i' = concat_{m=1..H} sigma( sum_j alpha_ij^(m) * W^(m) . h_j )

    HeteroConv runs one such GATv2 PER edge type r and sums over relations at
    each destination node:

        h_i' = sum_{r in relations(type(i))} GATv2_r( h_i, {h_j : j --r--> i} )
    """

    _account_in: Linear
    _featureless_embeds: ModuleDict
    _conv1: HeteroConv
    _conv2: HeteroConv
    _dropout: Dropout
    _head: Linear
    _featureless_types: tuple[NodeType, ...]

    def __init__(
        self,
        account_in_dim: int,
        edge_dim: int,
        hidden_dim: int = 64,
        heads: int = 4,
        dropout: float = 0.1,
        featureless_types: tuple[NodeType, ...] = ("Party", "Resolved_Entity"),
    ) -> None:
        super().__init__()
        self._featureless_types = featureless_types

        # Account input projection:  x_i = W_in . features_i + b
        #   features_i : [account_in_dim]            (= [31])
        #   W_in       : [hidden_dim, account_in_dim] (= [64, 31]; PyTorch stores
        #                Linear weight as [out, in] because y = x @ W^T + b)
        #   x_i        : [hidden_dim]                (= [64])
        self._account_in = Linear(account_in_dim, hidden_dim)

        # Featureless types (Party, Resolved_Entity) have no input features, so
        # each type gets ONE learned vector e_type in R^{hidden_dim}, broadcast
        # to all its nodes:  x_i = e_type  for every node i of that type.
        #   each Embedding(1, hidden_dim) weight : [1, 64]
        embeds: dict[str, Module] = {ntype: Embedding(1, hidden_dim) for ntype in featureless_types}
        self._featureless_embeds = ModuleDict(embeds)

        # With concat=True the H heads are concatenated, so each head must output
        # hidden_dim / H to reassemble to hidden_dim:
        #   per_head = hidden_dim / heads        (64 / 4 = 16)
        #   concat of H heads : H * per_head = hidden_dim  (4 * 16 = 64)
        # heads is a hyperparameter (must divide hidden_dim). More heads = more
        # independent attention patterns but thinner per-head dim; per_head ~ 8-16
        # is typical, so 4 or 8 heads suit hidden_dim = 64.
        if hidden_dim % heads != 0:
            raise ValueError(f"hidden_dim ({hidden_dim}) must be divisible by heads ({heads})")
        per_head = hidden_dim // heads

        # Two layers, because the sampler fetches 2 hops. Layer L lets each node
        # see L hops away: after conv1 a node summarizes its 1-hop neighborhood;
        # conv2 aggregates those summaries -> 2-hop reach.
        self._conv1 = self._make_conv(hidden_dim, per_head, heads, edge_dim, dropout)
        self._conv2 = self._make_conv(hidden_dim, per_head, heads, edge_dim, dropout)

        self._dropout = Dropout(dropout)

        # Readout head on Account vectors:  logit_i = w_out . h_i + b
        #   W_out : [1, hidden_dim] (= [1, 64]);  logit_i : scalar
        self._head = Linear(hidden_dim, 1)

    def _make_conv(
        self,
        in_dim: int,
        per_head: int,
        heads: int,
        edge_dim: int,
        dropout: float,
    ) -> HeteroConv:
        # One GATv2Conv per edge type. Each conv independently holds the W and a
        # of the attention equations (its own weights per relation), and each
        # runs `heads` parallel heads. Internally, per head:
        #     e_ij = a^T . LeakyReLU(W.h_i + W.h_j [+ W_e.edge_ij])
        #     alpha_ij = softmax_j(e_ij)
        #     head_out_i = sum_j alpha_ij * (W . h_j)        -> [per_head]
        # then concat over heads -> [heads * per_head] = [in_dim].
        convs: dict[EdgeType, MessagePassing] = {}
        for etype in _EDGE_TYPES:
            same = etype in _SAME_TYPE_EDGES
            convs[etype] = GATv2Conv(
                # (src_dim, dst_dim) bipartite in-channels; both = hidden_dim
                (in_dim, in_dim),
                # per-head output dim; concat of `heads` of these = hidden_dim
                per_head,
                heads=heads,  # H parallel attention heads
                concat=True,  # concat heads (-> hidden_dim), not average
                dropout=dropout,  # dropout on attention coefficients alpha_ij
                add_self_loops=same,  # include the W.h_i self term only for same-type edges
                # edge term W_e.edge_ij only on HAS_PAID, whose edge_attr is [E, edge_dim]=[E,34]
                edge_dim=edge_dim if etype == _HAS_PAID else None,
            )
        # aggr="sum": combine the per-relation results at each node by summation,
        #     h_i' = sum_r GATv2_r(...)
        return HeteroConv(convs, aggr="sum")

    def _initial_x(
        self, x_dict: dict[NodeType, Tensor], counts: dict[NodeType, int]
    ) -> dict[NodeType, Tensor]:
        out: dict[NodeType, Tensor] = {}

        # Account:  X_account @ W_in^T + b
        #   x_dict["Account"] : [N_account, 31]
        #   out["Account"]    : [N_account, 64]
        out[_ACCOUNT] = cast(Tensor, self._account_in(x_dict[_ACCOUNT]))

        # Featureless types: look up the single learned vector for every node.
        #   idx          : [n]   (all zeros -> the one embedding row)
        #   out[ntype]   : [n, 64]   (e_type broadcast to all n nodes)
        # idx must sit on the model's device (the Account projection already is),
        # or indexing the on-device embedding with a CPU index fails on MPS/CUDA.
        device = out[_ACCOUNT].device
        for ntype in self._featureless_types:
            n = counts.get(ntype, 0)
            embed = self._featureless_embeds[ntype]
            idx = torch.zeros(n, dtype=torch.long, device=device)
            out[ntype] = cast(Tensor, embed(idx))
        return out

    @override
    def forward(
        self,
        x_dict: dict[NodeType, Tensor],  # {"Account": [N,31], ...}
        edge_index_dict: dict[EdgeType, Tensor],  # each etype -> [2, E_etype] (COO)
        edge_attr_dict: dict[EdgeType, Tensor],  # {_HAS_PAID: [E_hp, 34]}
        node_counts: dict[NodeType, int],  # {"Account": N, "Party": ..., ...}
    ) -> Tensor:
        # Project / embed every node type to hidden_dim.
        #   h_dict[type] : [N_type, 64]
        h_dict: dict[NodeType, Tensor] = self._initial_x(x_dict, node_counts)

        # Layer 1: per-relation GATv2 attention, summed per node (1-hop mixing).
        #   for each type, h_i^(1) = sum_r sigma( sum_j alpha_ij^r * W^r . h_j )
        #   h1[type] : [N_type, 64]
        h1: dict[NodeType, Tensor] = cast(
            dict[NodeType, Tensor],
            self._conv1(h_dict, edge_index_dict, edge_attr_dict),
        )
        # ELU nonlinearity:  ELU(x) = x if x > 0 else (exp(x) - 1)
        h1 = {k: F.elu(v) for k, v in h1.items()}
        # Dropout (training only): randomly zero a fraction of activations.
        h1 = {k: cast(Tensor, self._dropout(v)) for k, v in h1.items()}

        # Layer 2: aggregates the 1-hop summaries -> each node reflects 2 hops.
        #   h2[type] : [N_type, 64]
        h2: dict[NodeType, Tensor] = cast(
            dict[NodeType, Tensor],
            self._conv2(h1, edge_index_dict, edge_attr_dict),
        )
        h2 = {k: F.elu(v) for k, v in h2.items()}

        # Readout on accounts only:  logit_i = w_out . h_i + b
        #   account_h : [N_account, 64]
        #   logits    : [N_account, 1] -> reshape -> [N_account]
        account_h = h2[_ACCOUNT]
        logits: Tensor = cast(Tensor, self._head(account_h))
        return logits.reshape(-1)

## 23 · The backend — wiring it together
`pyg/backend.py`

`TigerGraphRemoteBackend` owns the single client and the one shared mapper, and builds the sampler, feature store, graph store, and the **NodeLoader** from them. This is the object that makes 'a PyG loader, powered by TigerGraph' actually consistent.

In [26]:
import torch
from torch import Tensor
from torch_geometric.loader import NodeLoader



class TigerGraphRemoteBackend:
    """
    Shared container wiring TigerGraph into PyG's remote-backend interfaces.

    Holds the single TigerGraph client and the one persistent NodeIDMapper that
    the sampler and the feature store must agree on: the sampler registers each
    sampled node's global string id and writes the assigned integer into PyG's
    node tensor, and the feature store later reverses those same integers back
    to string ids to fetch features. Because PyG's _get_tensor receives only
    (group_name, attr_name, index) and not the sampler's metadata, this mapping
    has to live on a shared object both sides reference (the FalkorDB pattern).

    The mapper persists across the run, so it grows with the set of distinct
    nodes actually sampled (bounded by the labeled seeds and their sampled
    neighborhoods), not the full graph. For true billion-scale with a fixed
    memory ceiling, the mapper could later evict stale batches (LRU); that is a
    future optimization and not required at the current scale.
    """

    _client: Client
    _mapper: NodeIDMapper

    def __init__(self, client: Client) -> None:
        self._client = client
        self._mapper = NodeIDMapper()

    @property
    def client(self) -> Client:
        """The shared TigerGraph client used for sampling and feature fetches."""
        return self._client

    @property
    def mapper(self) -> NodeIDMapper:
        """The shared, persistent global-id <-> integer mapper."""
        return self._mapper

    def make_sampler(
        self,
        seed_ids: tuple[str, ...],
        fanout: NeighborFanout | None = None,
        query_name: str = "sample_khop_neighborhood",
        allow_val: bool = True,
        allow_test: bool = True,
    ) -> TigerGraphHeteroSampler:
        """
        Build a k-hop sampler bound to this backend's client and mapper.

        The sampler shares this backend's mapper, so the integers it writes into
        PyG node tensors are reversible by the feature store built from the same
        backend.

        allow_val / allow_test control strict-inductive split filtering in the
        sampler query: when False, neighborhoods do not traverse into val / test
        accounts (so their features cannot leak into a seed's embedding). Train
        loaders pass both False; val loaders pass allow_val=True, allow_test=
        False; test loaders pass both True.
        """
        return TigerGraphHeteroSampler(
            client=self._client,
            seed_ids=seed_ids,
            mapper=self._mapper,
            fanout=fanout,
            query_name=query_name,
            allow_val=allow_val,
            allow_test=allow_test,
        )

    def make_feature_store(
        self, normalizer: FeatureNormalizer | None = None
    ) -> TigerGraphFeatureStore:
        """
        Build a feature store bound to this backend's client and mapper.

        The store shares this backend's mapper, so it reverses the same integer
        ids the sampler assigned back to global string ids when fetching
        features. This shared mapper is why the sampler and store must come from
        one backend (PyG's _get_tensor sees only the integers, not the sampler's
        metadata).

        normalizer, if given, standardizes account features as they are served
        (fit on the train split; see FeatureNormalizer).
        """
        return TigerGraphFeatureStore(
            client=self._client, mapper=self._mapper, normalizer=normalizer
        )

    def make_graph_store(self) -> TigerGraphGraphStore:
        """
        Build a graph store bound to this backend's client and mapper.

        The store shares this backend's mapper so any edge indices it exports
        use the same integer id space as the sampler and feature store. Note the
        sampler, not this store, drives batch sampling; the store serves whole
        edge types on demand (off the hot path).
        """
        return TigerGraphGraphStore(client=self._client, mapper=self._mapper)

    def remote_backend(
        self,
    ) -> tuple[TigerGraphFeatureStore, TigerGraphGraphStore]:
        """Return the (feature_store, graph_store) pair PyG's NodeLoader wants.

        Both share this backend's client and mapper, so the data=(feature_store,
        graph_store) tuple passed to a loader is internally consistent.
        """
        return (self.make_feature_store(), self.make_graph_store())

    def make_loader(
        self,
        seed_ids: tuple[str, ...],
        reference_epoch_s: float,
        max_bins: int,
        fanout: NeighborFanout | None = None,
        batch_size: int = 1,
        shuffle: bool = False,
        allow_val: bool = True,
        allow_test: bool = True,
        normalizer: FeatureNormalizer | None = None,
    ) -> NodeLoader:
        """
        Build a PyG NodeLoader that yields HeteroData batches from TigerGraph.

        Wires the (feature_store, graph_store) pair, the k-hop sampler, and the
        HAS_PAID edge-feature transform, all sharing this backend's client and
        mapper. input_nodes is the Account seed set as integer indices into
        seed_ids; the sampler maps those to global ids before sampling.

        Node features arrive via the feature store; HAS_PAID edge features are
        attached by the transform, since PyG's remote feature store serves node
        features only.

        allow_val / allow_test enforce strict-inductive sampling (see
        make_sampler). For leakage-free training pass allow_val=False,
        allow_test=False; for validation pass allow_val=True, allow_test=False;
        for test pass both True.
        """
        feature_store = self.make_feature_store(normalizer=normalizer)
        graph_store = self.make_graph_store()
        sampler = self.make_sampler(
            seed_ids=seed_ids,
            fanout=fanout,
            allow_val=allow_val,
            allow_test=allow_test,
        )
        transform = HasPaidEdgeFeatureAttacher(
            client=self._client,
            mapper=self._mapper,
            reference_epoch_s=reference_epoch_s,
            max_bins=max_bins,
        )
        seed_index: Tensor = torch.arange(len(seed_ids), dtype=torch.long)
        return NodeLoader(
            data=(feature_store, graph_store),
            node_sampler=sampler,
            input_nodes=("Account", seed_index),
            transform=transform,
            batch_size=batch_size,
            shuffle=shuffle,
        )

## 24 · Seed pool + positive-anchored batches
`training/seeds.py`

`SeedPool` splits seeds into positives vs unlabelled. `epoch_batches` yields one epoch of batches that put a **fixed number of positives in every batch** (sampled with replacement, since positives are scarce) — this is what guarantees nnPU's positive term always has something to average over. It **raises** if there are no positives.

In [27]:
from collections.abc import Iterator, Sequence
import random


class SeedPool:
    """
    Account-id pools for one split, partitioned by PU label.

    Separates the few revealed positives (pu_label == 1) from the unlabeled
    majority (pu_label == 0), because the nnPU loss needs positives present in
    every batch (its positive_risk term is estimated only from positives). The
    ids come from the caller (read from the graph), so this holds no I/O.

    positives:  ids with pu_label == 1
    unlabeled:  ids with pu_label == 0
    """

    positives: tuple[str, ...]
    unlabeled: tuple[str, ...]

    def __init__(self, positives: Sequence[str], unlabeled: Sequence[str]) -> None:
        self.positives = tuple(positives)
        self.unlabeled = tuple(unlabeled)

    @property
    def num_positives(self) -> int:
        return len(self.positives)

    @property
    def num_unlabeled(self) -> int:
        return len(self.unlabeled)


def epoch_batches(
    pool: SeedPool,
    batch_size: int,
    positives_per_batch: int,
    seed: int,
) -> Iterator[tuple[str, ...]]:
    """
    Yield one epoch of seed-id batches that oversample the rare positives.

    Each batch contains exactly positives_per_batch positive seeds (sampled WITH
    replacement, since there may be fewer unique positives than batches need)
    plus (batch_size - positives_per_batch) unlabeled seeds (sampled WITHOUT
    replacement within the epoch, so the unlabeled set is covered once). The
    number of batches is set so the unlabeled pool is consumed once per epoch:
    n_batches = ceil(num_unlabeled / unlabeled_per_batch).

    This guarantees nnPU's positive_risk always has positives to average over,
    while still traversing the unlabeled data once per epoch. Each yielded tuple
    is the seed_ids argument for backend.make_loader.

    Raises ValueError if positives_per_batch is not in [1, batch_size), or if
    the pool has no positives (nnPU cannot train without them).
    """
    if positives_per_batch < 1:
        raise ValueError("positives_per_batch must be >= 1 for nnPU.")
    if positives_per_batch >= batch_size:
        raise ValueError("positives_per_batch must be < batch_size.")
    if pool.num_positives == 0:
        raise ValueError("seed pool has no positives; nnPU cannot train.")

    rng = random.Random(seed)
    unlabeled_per_batch = batch_size - positives_per_batch

    # shuffle the unlabeled pool once; we consume it in order, one pass per epoch
    unlabeled = list(pool.unlabeled)
    rng.shuffle(unlabeled)

    if pool.num_unlabeled == 0:
        n_batches = 1
    else:
        n_batches = (pool.num_unlabeled + unlabeled_per_batch - 1) // unlabeled_per_batch

    positives = list(pool.positives)

    for b in range(n_batches):
        # positives: sample WITH replacement (few uniques, need them every batch)
        pos_seeds = [positives[rng.randrange(len(positives))] for _ in range(positives_per_batch)]

        # unlabeled: take the next slice of the shuffled pool (without replacement)
        start = b * unlabeled_per_batch
        unl_seeds = unlabeled[start : start + unlabeled_per_batch]

        batch = pos_seeds + unl_seeds
        rng.shuffle(batch)  # interleave so order doesn't encode the label
        yield tuple(batch)

## 25 · Read seeds from the graph
`training/seeds_source.py`

`fetch_split_seeds` pulls a split's account ids and their `pu_label` straight from the graph.

In [28]:
from typing import cast


_QUERY_NAME = "get_split_accounts"
_COL_ACCOUNT_ID = "account_id"
_COL_PU_LABEL = "pu_label"


class SplitSeeds:
    """
    One split's seed account ids, plus an in-memory lookup of each account's
    pu_label attribute (1 = revealed positive, 0 = unlabeled), read from the graph.

    pu_label lives on the account vertex; this just caches it by id so the
    training loop reads labels from memory instead of re-querying per batch.
    """

    account_ids: tuple[str, ...]
    pu_label_of: dict[str, int]

    def __init__(self, account_ids: tuple[str, ...], pu_label_of: dict[str, int]) -> None:
        self.account_ids = account_ids
        self.pu_label_of = pu_label_of

    @property
    def num_positives(self) -> int:
        return sum(1 for v in self.pu_label_of.values() if v == 1)

    @property
    def num_unlabeled(self) -> int:
        return sum(1 for v in self.pu_label_of.values() if v == 0)


def _parse_rows(raw: list[object]) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    for block in raw:
        if not (isinstance(block, dict) and "result" in block):
            continue
        vertices = cast(dict[str, object], block)["result"]
        if not isinstance(vertices, list):
            continue
        for vertex in cast(list[object], vertices):
            if not isinstance(vertex, dict):
                continue
            attrs = cast(dict[str, object], vertex).get("attributes")
            if isinstance(attrs, dict):
                rows.append(cast(dict[str, object], attrs))
    return rows


def fetch_split_seeds(client: Client, split: str) -> SplitSeeds:
    """
    Fetch one split's accounts + pu_label from the graph.

    split is "train" | "val" | "test". Runs get_split_accounts and returns the
    seed ids plus the pu_label lookup the trainer feeds to the nnPU loss.
    """
    raw = cast(
        list[object],
        client.conn.runInstalledQuery(_QUERY_NAME, {"split": split}),
    )
    rows = _parse_rows(raw)
    account_ids: list[str] = []
    pu_label_of: dict[str, int] = {}
    for attrs in rows:
        account_id = str(attrs[_COL_ACCOUNT_ID])
        account_ids.append(account_id)
        pu_label_of[account_id] = int(cast(int, attrs[_COL_PU_LABEL]))
    return SplitSeeds(account_ids=tuple(account_ids), pu_label_of=pu_label_of)

## 26 · nnPU loss
`training/loss.py`

`NonNegativePULoss` (Kiryo 2017). Estimates the negatives' risk **indirectly** from positives + the known prior π, instead of calling unlabelled accounts negative. When the estimated negative risk goes below zero (overfitting), it flips to push that risk back up — the non-negative correction. `positive_weight` is a separate knob that keeps the positive term from starving under extreme imbalance, without biasing the estimator.

In [29]:
from typing import override

import torch
from torch import Tensor
from torch.nn import Module


class NonNegativePULoss(Module):
    """
    Non-negative PU loss for positive-unlabeled binary classification
    (Kiryo et al., 2017).

    Standard cross-entropy is wrong here: the unlabeled set is a MIXTURE of true
    negatives and undiscovered positives (hidden mules), so labelling all
    unlabeled as negative teaches the model to suppress the very signal we want.
    PU learning instead estimates the negative risk indirectly from positives
    and unlabeled, using the known class prior pi.

    With surrogate loss l(z) = sigmoid(-z) (non-increasing), per-sample:
        l_pos_i = l(+f_i) = sigmoid(-f_i)     # loss if i treated as POSITIVE
        l_neg_i = l(-f_i) = sigmoid(+f_i)     # loss if i treated as NEGATIVE

    Risks (n_p = #positives, n_u = #unlabeled in the batch):
        R_p^+   = (1/n_p) * sum_{i in P} l_pos_i           # positives, as positive
        R_p^-   = (1/n_p) * sum_{i in P} l_neg_i           # positives, as negative
        R_u^-   = (1/n_u) * sum_{i in U} l_neg_i           # unlabeled, as negative

        positive_risk = pi * R_p^+
        negative_risk = R_u^-  -  pi * R_p^-     # unbiased estimate of the
                                                 # negatives-as-negative risk

    Objective:
        uPU:   R = positive_risk + negative_risk
        nnPU:  if negative_risk >= -beta:  R = positive_risk + negative_risk
               else (risk went negative -> overfitting):
                       minimize  gamma * (-negative_risk)
                       i.e. push the negative risk back up toward 0.

    The loss returned for backprop follows the nnPU rule; the always-unclamped
    value is also reported for monitoring.

    Args:
        prior: pi = P(y = 1), the assumed TRUE fraction of positives in the
            population (the known simulation mule rate; NOT the revealed-label
            rate, which masking makes artificially low). Must be in (0, 1).
        beta: lower bound the estimated negative risk is clamped against. The
            paper fixes beta = 0.
        gamma: scale on the negative-risk gradient when the correction fires.
            The paper fixes gamma = 1.
        positive_weight: weight on the positive_risk term ONLY. Defaults to
            prior, which reproduces the textbook nnPU objective exactly. Under
            extreme imbalance (pi ~ 0.003) the textbook weight makes
            positive_risk a negligible fraction of the loss (here ~0.001), so
            the optimizer minimizes the unlabeled term alone by driving every
            logit down -- dragging positives BELOW unlabeled and inverting the
            ranking. Raising positive_weight (e.g. to pi-balanced 0.5, or any
            value >> pi) restores gradient signal to the positives WITHOUT
            touching the prior used in the unbiased negative-risk correction
            below, so that estimator stays unbiased. This separation is the fix:
            it reweights how much positives matter, not the statistics of the
            negatives-as-negative estimate. Must be in (0, 1).

    Convention: targets t use +1 for revealed positives and 0 for unlabeled
    (matching pu_label in this project). f are raw logits, shape [N].
    """

    _prior: float
    _beta: float
    _gamma: float
    _positive_weight: float

    def __init__(
        self,
        prior: float,
        beta: float = 0.0,
        gamma: float = 1.0,
        positive_weight: float | None = None,
    ) -> None:
        super().__init__()
        if not (0.0 < prior < 1.0):
            raise ValueError("prior (class prior pi) must be in (0, 1).")
        if gamma <= 0.0:
            raise ValueError("gamma must be positive.")
        # Default to prior so the objective is byte-for-byte the textbook nnPU
        # unless the caller deliberately opts into reweighting.
        weight = prior if positive_weight is None else positive_weight
        if not (0.0 < weight < 1.0):
            raise ValueError("positive_weight must be in (0, 1).")
        self._prior = prior
        self._beta = beta
        self._gamma = gamma
        self._positive_weight = weight

    @staticmethod
    def _surrogate_pos(logits: Tensor) -> Tensor:
        # l(+f) = sigmoid(-f): small when f is large/positive (confident positive)
        return torch.sigmoid(-logits)

    @staticmethod
    def _surrogate_neg(logits: Tensor) -> Tensor:
        # l(-f) = sigmoid(+f): small when f is large/negative (confident negative)
        return torch.sigmoid(logits)

    @override
    def forward(self, logits: Tensor, targets: Tensor) -> tuple[Tensor, Tensor]:
        # logits: [N] raw scores f_i ; targets: [N] in {0 unlabeled, 1 positive}
        positive = (targets == 1).to(logits.dtype)  # [N] 1.0 at positives
        unlabeled = (targets == 0).to(logits.dtype)  # [N] 1.0 at unlabeled

        # guard against empty groups in a batch (avoid divide-by-zero)
        n_positive = torch.clamp(positive.sum(), min=1.0)
        n_unlabeled = torch.clamp(unlabeled.sum(), min=1.0)

        l_pos = self._surrogate_pos(logits)  # [N]
        l_neg = self._surrogate_neg(logits)  # [N]

        # positive_risk = positive_weight * (1/n_p) * sum_{P} l(+f)
        # positive_weight defaults to prior (textbook nnPU); raise it to keep the
        # positives from being weighted into irrelevance under extreme imbalance.
        positive_risk = self._positive_weight * torch.sum(positive * l_pos) / n_positive

        # negative_risk = (1/n_u) sum_{U} l(-f)  -  pi * (1/n_p) sum_{P} l(-f)
        # NOTE: this term keeps the TRUE prior, not positive_weight -- it is the
        # unbiased estimate of the negatives-as-negative risk and must use pi.
        negative_risk = (
            torch.sum(unlabeled * l_neg) / n_unlabeled
            - self._prior * torch.sum(positive * l_neg) / n_positive
        )

        objective = positive_risk + negative_risk

        # nnPU correction: if the estimated negative risk dips below -beta, the
        # unbiased objective is overfitting; replace the backprop target with the
        # gradient-ascending term gamma * (-negative_risk) to push it back up.
        if negative_risk.item() < -self._beta:
            train_loss = self._gamma * (-negative_risk)
        else:
            train_loss = objective

        return train_loss, objective

## 27 · Evaluation metrics
`training/metrics.py`

The PU-correct scoreboard: **Proxy AUC** for model selection, **hidden-recall@k** (with the achievable **ceiling**), and the **gains/lift** sweep across alert rates. Plain accuracy is deliberately not the headline — under 1% prevalence it rewards a model that flags nobody.

In [30]:
from collections.abc import Sequence
from dataclasses import dataclass
from enum import IntEnum
from typing import cast

import numpy as np
from numpy.typing import NDArray
from sklearn.metrics import average_precision_score, roc_auc_score


def _int(value: np.intp | int) -> int:
    # numpy reductions / shapes are typed Any under strict checking; a cast at
    # the call site (cast(np.intp, ...)) erases the Any, and this funnels the
    # concrete numpy integer to a Python int in one place.
    return int(value)


def _count(array: NDArray[np.bool_] | NDArray[np.int64]) -> int:
    # number of truthy / nonzero entries, laundered to a plain int
    return _int(cast("np.intp", array.sum()))


def _length(array: NDArray[np.float64] | NDArray[np.int64]) -> int:
    return _int(cast("int", array.shape[0]))


def _precision(scores: NDArray[np.float64], labels: NDArray[np.int64], k: int) -> float:
    # fraction of the top-k highest-scored accounts that are positive
    if k <= 0:
        return 0.0
    order = np.argsort(-scores)
    top = order[:k]
    hits = _count(labels[top])
    denom = min(k, _length(labels))
    return hits / float(denom)


# ââ PRODUCTION metrics: computed against pu_label only ââ
# These use only the labels the model also trains on (pu_label: 1 = known
# positive, 0 = unlabeled), so they exist unchanged on real data. This is what
# drives early stopping and what you would report in production.


@dataclass(frozen=True, slots=True)
class ValScores:
    """
    Validation ranking quality, computed against pu_label (production-valid).

    roc_auc is the model-selection number and the early-stop signal. In PU data
    the unlabeled validation points are treated as (corrupted) negatives, and
    ROC-AUC computed that way is the Proxy AUC (PAUC): ranking under the true
    labels would order two classifiers the same way PAUC does, because AUC is
    provably robust to this label corruption -- a classifier with higher PAUC
    has higher true AUC. (Sakai et al.; PU model-selection literature.) That is
    why early stopping uses roc_auc, NOT average_precision: AP is sensitive to
    the unlabeled-as-negative assumption (the unlabeled set still contains real
    positives, which depress precision unpredictably), so it is reported as a
    diagnostic only, not optimized against. precision_at_k mirrors an
    investigation queue (of the top-k flagged, the fraction that are labelled
    positives) and is likewise diagnostic.

    Every field here derives from pu_label, so nothing in this object depends on
    knowing the true (hidden) mules -- it is identical in production.
    """

    average_precision: float
    roc_auc: float
    precision_at_k: float
    k: int
    num_evaluated: int
    num_labeled_positives: int


def evaluate_ranking(scores: NDArray[np.float64], pu_label: NDArray[np.int64], k: int) -> ValScores:
    """
    Score model output against pu_label (the labels the model also trains on).

    scores:   model output per account (higher = more mule-like); shape [M].
    pu_label: 1 if the account is a known/labelled positive, else 0; shape [M].
    k:        cutoff for precision_at_k.

    average_precision / roc_auc are computed against pu_label; precision_at_k
    uses the score ranking. roc_auc is the Proxy AUC used for model selection
    (see ValScores); average_precision is diagnostic. Raises ValueError on shape
    mismatch or if pu_label has only one class (ranking metrics undefined).
    Production-valid: uses no ground-truth / hidden-mule information.
    """
    if scores.shape != pu_label.shape:
        raise ValueError("scores and pu_label must have the same shape.")
    if scores.ndim != 1:
        raise ValueError("scores must be 1-D.")

    n_total = _length(pu_label)
    n_pos = _count(pu_label)
    if n_pos == 0 or n_pos == n_total:
        raise ValueError(
            f"pu_label must contain both classes; got {n_pos} positives of {n_total}. "
            + "PU model selection (Proxy AUC) needs at least one revealed positive in "
            + "the evaluated split; train.py guards this before training, so reaching "
            + "here means the split lost its positives."
        )

    ap = float(average_precision_score(pu_label, scores))
    auc = float(roc_auc_score(pu_label, scores))
    p_at_k = _precision(scores, pu_label, k)

    return ValScores(
        average_precision=ap,
        roc_auc=auc,
        precision_at_k=p_at_k,
        k=k,
        num_evaluated=n_total,
        num_labeled_positives=n_pos,
    )


# ââ SYNTHETIC-ONLY evaluation: requires the answer key (true_label, bucket) ââ
# This is NOT part of training. It exists to measure, on synthetic data where
# ground truth is known, whether the model generalizes to mules it was never
# told about. On real data there is no answer key, so this is never run and
# these symbols are never imported by the training loop.


class Bucket(IntEnum):
    """Per-account evaluation bucket (synthetic answer key; from the masking step)."""

    UNLABELED_NEG = 0
    REVEALED_POS = 1
    HIDDEN_POS = 2


@dataclass(frozen=True, slots=True)
class HiddenScores:
    """
    Generalization measure for the PU experiment (synthetic data only).

    hidden_recall_at_k is the key number: among the HIDDEN mules (truly mules
    but never labelled to the model), the fraction the model ranks in its top-k.
    A model that merely memorized the few revealed positives scores low here.
    average_precision_true / roc_auc_true recompute ranking quality against the
    TRUE labels (not pu_label), so they reveal how well the model found the
    unlabelled mules too. All of this depends on the synthetic answer key.
    """

    hidden_recall_at_k: float
    average_precision_true: float
    roc_auc_true: float
    k: int
    num_hidden_positives: int
    num_true_positives: int


def _hidden_recall(scores: NDArray[np.float64], is_hidden: NDArray[np.bool_], k: int) -> float:
    # of all HIDDEN positives, the fraction ranked within the top-k overall
    n_hidden = _count(is_hidden)
    if k <= 0 or n_hidden == 0:
        return 0.0
    order = np.argsort(-scores)
    top = order[:k]
    hidden_in_top = _count(is_hidden[top])
    return hidden_in_top / float(n_hidden)


def evaluate_hidden(
    scores: NDArray[np.float64],
    true_label: NDArray[np.int64],
    bucket: NDArray[np.int64],
    k: int,
) -> HiddenScores:
    """
    Score generalization to hidden mules, using the synthetic answer key.

    scores:     model output per account (higher = more mule-like); shape [M].
    true_label: 1 if the account is truly a mule, else 0; shape [M].
    bucket:     REVEALED_POS / HIDDEN_POS / UNLABELED_NEG per account; shape [M].
    k:          cutoff for hidden_recall_at_k.

    SYNTHETIC ONLY: requires ground truth that does not exist in production. Run
    this as a separate evaluation step, never inside the training loop.
    """
    if not (scores.shape == true_label.shape == bucket.shape):
        raise ValueError("scores, true_label, and bucket must have the same shape.")
    if scores.ndim != 1:
        raise ValueError("scores must be 1-D.")

    n_total = _length(true_label)
    n_pos = _count(true_label)
    if n_pos == 0 or n_pos == n_total:
        raise ValueError(
            f"true_label must contain both classes; got {n_pos} positives of {n_total}."
        )

    is_hidden: NDArray[np.bool_] = np.equal(bucket, int(Bucket.HIDDEN_POS))

    return HiddenScores(
        hidden_recall_at_k=_hidden_recall(scores, is_hidden, k),
        average_precision_true=float(average_precision_score(true_label, scores)),
        roc_auc_true=float(roc_auc_score(true_label, scores)),
        k=k,
        num_hidden_positives=_count(is_hidden),
        num_true_positives=n_pos,
    )


# ââ SYNTHETIC-ONLY: multi-k recall + ceiling-normalized recall ââ
# Extends evaluate_hidden's single-k report to several review-queue depths and
# reports, alongside each raw recall, the FRACTION OF THE ACHIEVABLE CEILING it
# captures. The ceiling exists because recall@k is capped at k / n_hidden when
# k < n_hidden: with k slots and more than k hidden mules, a perfect ranker
# still cannot place all hidden mules in the top k. The normalized number
# divides observed recall by that cap, so 1.0 means "ranked as many hidden
# mules into the top k as is mathematically possible," NOT "found every mule."
# It is a diagnostic that separates model ranking quality from the keyhole
# effect of a small k; the production metric remains recall at the single k an
# analyst can actually review.

_DEFAULT_KS: tuple[int, ...] = (100, 250, 500, 1000)


def recall_ceiling_at_k(n_hidden: int, k: int) -> float:
    """Maximum achievable hidden-recall at cutoff k.

    A perfect ranker puts every hidden mule above every negative, so it captures
    min(k, n_hidden) of them; dividing by n_hidden gives the ceiling. Equals
    min(1.0, k / n_hidden). Returns 0.0 when there are no hidden mules (recall is
    undefined) and 0.0 for non-positive k.
    """
    if n_hidden <= 0 or k <= 0:
        return 0.0
    return min(1.0, k / float(n_hidden))


@dataclass(frozen=True, slots=True)
class RecallAtK:
    """One (k, recall, ceiling, normalized) row of the multi-k report.

    recall:
        fraction of ALL hidden mules ranked within the top k (the same quantity
        evaluate_hidden reports at its single k).
    ceiling:
        min(1.0, k / num_hidden_positives) -- the most recall any ranker can
        achieve at this k. Below n_hidden the top-k simply has too few slots.
    normalized_recall:
        recall / ceiling, clamped to [0, 1]; the share of the achievable maximum
        actually captured. Read as ranking quality at this depth, not as
        coverage of the true mule population.
    hidden_in_top_k:
        raw count of hidden mules in the top k (recall * num_hidden_positives),
        surfaced so the report can show "37 of 169 hidden mules in the top 250".
    """

    k: int
    recall: float
    ceiling: float
    normalized_recall: float
    hidden_in_top_k: int


@dataclass(frozen=True, slots=True)
class HiddenScoresMultiK:
    """evaluate_hidden, reported across several review-queue depths.

    rows holds one RecallAtK per requested k (ascending). primary_k names the k
    treated as the production / headline number (the depth an analyst can review)
    so a caller can highlight it; the others are diagnostic context. The TRUE-label
    ranking metrics (average_precision_true, roc_auc_true) are k-independent and
    are reported once, identical to evaluate_hidden's.
    """

    rows: tuple[RecallAtK, ...]
    primary_k: int
    average_precision_true: float
    roc_auc_true: float
    num_hidden_positives: int
    num_true_positives: int

    def row_for(self, k: int) -> RecallAtK:
        for row in self.rows:
            if row.k == k:
                return row
        raise KeyError(f"no RecallAtK row for k={k}; have {[r.k for r in self.rows]}")

    @property
    def primary(self) -> RecallAtK:
        return self.row_for(self.primary_k)


def evaluate_hidden_multi_k(
    scores: NDArray[np.float64],
    true_label: NDArray[np.int64],
    bucket: NDArray[np.int64],
    ks: Sequence[int] = _DEFAULT_KS,
    primary_k: int = 100,
) -> HiddenScoresMultiK:
    """Multi-k generalization report for the PU experiment (synthetic only).

    Computes hidden-recall at each k in ks, plus the ceiling and the
    ceiling-normalized recall at that k, and reports the k-independent TRUE-label
    AP / ROC-AUC once. Sorting the scores once and reusing the order keeps this
    O(n log n + sum_k k) rather than re-sorting per k.

    scores:     model output per account (higher = more mule-like); shape [M].
    true_label: 1 if the account is truly a mule, else 0; shape [M].
    bucket:     REVEALED_POS / HIDDEN_POS / UNLABELED_NEG per account; shape [M].
    ks:         review-queue depths to report (deduped, sorted ascending). A k
                larger than M is kept but its top-k is the whole array, so recall
                there is 1.0 and the ceiling is 1.0.
    primary_k:  the k flagged as the headline/production number. Must be one of
                the (deduplicated) ks, so HiddenScoresMultiK.primary is well
                defined; a primary_k absent from ks is a caller error.

    SYNTHETIC ONLY: requires the answer key (true_label, bucket) that does not
    exist in production. Never call inside the training loop.
    """
    if not (scores.shape == true_label.shape == bucket.shape):
        raise ValueError("scores, true_label, and bucket must have the same shape.")
    if scores.ndim != 1:
        raise ValueError("scores must be 1-D.")

    n_total = _length(true_label)
    n_pos = _count(true_label)
    if n_pos == 0 or n_pos == n_total:
        raise ValueError(
            f"true_label must contain both classes; got {n_pos} positives of {n_total}."
        )

    # dedupe + sort so rows are stable and a repeated k is not double-reported
    sorted_ks = tuple(sorted({int(k) for k in ks}))
    if not sorted_ks:
        raise ValueError("ks must contain at least one cutoff.")
    if not all(k > 0 for k in sorted_ks):
        raise ValueError(f"all ks must be positive; got {sorted_ks}.")
    if primary_k not in sorted_ks:
        raise ValueError(
            f"primary_k ({primary_k}) must be one of ks ({sorted_ks}); it is the "
            + "headline depth and must have a computed row."
        )

    is_hidden: NDArray[np.bool_] = np.equal(bucket, int(Bucket.HIDDEN_POS))
    n_hidden = _count(is_hidden)

    # sort once (descending score); slice prefixes for each k
    order = np.argsort(-scores)
    hidden_sorted = is_hidden[order]
    # cumulative count of hidden mules down the ranking; cumulative[t-1] is the
    # number of hidden mules within the top t
    cumulative_hidden = np.cumsum(hidden_sorted.astype(np.int64))

    rows: list[RecallAtK] = []
    for k in sorted_ks:
        top = min(k, n_total)
        hidden_in_top = _int(cast("np.intp", cumulative_hidden[top - 1])) if top > 0 else 0
        recall = hidden_in_top / float(n_hidden) if n_hidden > 0 else 0.0
        ceiling = recall_ceiling_at_k(n_hidden, k)
        normalized = min(1.0, recall / ceiling) if ceiling > 0.0 else 0.0
        rows.append(
            RecallAtK(
                k=k,
                recall=recall,
                ceiling=ceiling,
                normalized_recall=normalized,
                hidden_in_top_k=hidden_in_top,
            )
        )

    return HiddenScoresMultiK(
        rows=tuple(rows),
        primary_k=primary_k,
        average_precision_true=float(average_precision_score(true_label, scores)),
        roc_auc_true=float(roc_auc_score(true_label, scores)),
        num_hidden_positives=n_hidden,
        num_true_positives=n_pos,
    )


def format_hidden_multi_k(report: HiddenScoresMultiK) -> str:
    """Render a HiddenScoresMultiK as an aligned text block for logs/console.

    Mirrors the existing evaluate_hidden printout style: a small header with the
    k-independent figures, then one line per k. The normalized column is labelled
    'recall/ceil' and the header states explicitly that 1.0 means 'best possible
    at this k', so 0.55 is never misread as 'found 55% of all mules'.
    """
    lines: list[str] = []
    lines.append("=" * 60)
    lines.append("HIDDEN-MULE GENERALIZATION (multi-k)")
    lines.append("=" * 60)
    lines.append(f"  true mules             : {report.num_true_positives}")
    lines.append(f"  hidden mules           : {report.num_hidden_positives}")
    lines.append(f"  AP (vs true labels)    : {report.average_precision_true:.4f}")
    lines.append(f"  AUC (vs true labels)   : {report.roc_auc_true:.4f}")
    lines.append("")
    lines.append("  recall/ceil = share of the BEST POSSIBLE recall at that k")
    lines.append("  (1.0 = every hidden mule that could fit in the top-k is there,")
    lines.append("   NOT that every mule in the bank was found)")
    lines.append("")
    header = (
        f"  {'k':>6}  {'recall':>8}  {'ceiling':>8}  {'recall/ceil':>12}  {'hidden_in_top_k':>16}"
    )
    lines.append(header)
    for row in report.rows:
        star = "  <- primary" if row.k == report.primary_k else ""
        lines.append(
            f"  {row.k:>6}  {row.recall:>8.4f}  {row.ceiling:>8.4f}  "
            + f"{row.normalized_recall:>12.4f}  "
            + f"{row.hidden_in_top_k:>6} / {report.num_hidden_positives:<7}{star}"
        )
    lines.append("=" * 60)
    return "\n".join(lines)


# ââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââ
# CAPACITY-SWEEP (gains / lift) EVALUATION
# ââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââ
# Operating-point metrics reported across a RANGE of review capacities rather
# than one arbitrary k. Each capacity is an alert rate (fraction of accounts an
# analyst team reviews), because in real AML programs the cutoff is set by
# staffing, not by a statistic -- and no single rate is "correct" for a summary report
# on synthetic data, so we report the whole curve and let each viewer read off
# their own column.
#
# Real-world anchors (for sizing the default sweep and the reference line):
#   * Production bank alert rates are tiny: a multi-bank survey saw ~0.04% of
#     transactions alerted. Account-level review budgets are larger but still
#     well under a few percent.
#   * Genuine-suspicion base rates run ~1-5%, sometimes <1%.
#   * A skilled human investigator's referrals convert to a SAR ~40-50% of the
#     time -- the realistic precision ceiling to benchmark against.
#
# WHICH METRICS, AND WHY THESE:
#   recall    -- the ONLY operating-point metric provably estimable on PU data
#                (under the standard PU assumption precision is biased, so recall
#                is the unattackable detection number).
#   precision -- reported because THIS dataset is synthetic and carries a true-
#                label answer key; on real PU data it would be biased downward
#                (correctly-flagged hidden positives counted as false positives).
#                Always present it flagged as synthetic-only.
#   lift      -- precision divided by the population base rate: "Nx better than
#                random at this depth". Imbalance-native and intuitive.
#   alert_rate/k -- the operating context that ties everything to staffing and
#                kills the "arbitrary k" objection.
#
# All four are computed against TRUE labels here (synthetic answer key). On real
# data you would compute recall/lift against revealed positives and treat them
# as PU estimates; precision would not be trustworthy. This module does not hide
# that -- the formatter prints the caveat.

# Default capacity sweep: fractions of the scored population to review. Brackets
# the realistic range from "true bank alert rate" (~0.05%) up to a "generous
# review budget" (2%). Override per audience.
_DEFAULT_ALERT_RATES: tuple[float, ...] = (0.0005, 0.001, 0.005, 0.01, 0.02)

# Reference precision a strong human analyst achieves (SAR conversion ~40-50%).
# Drawn on the precision curve as a benchmark line; not a model output.
_HUMAN_ANALYST_PRECISION: float = 0.45


@dataclass(frozen=True, slots=True)
class GainsRow:
    """One operating point in a capacity sweep, against TRUE labels.

    alert_rate:
        fraction of the population reviewed at this cutoff (the x-axis).
    k:
        number of accounts reviewed = round(alert_rate * n_total); the top-k by
        score. Reported alongside alert_rate so the audience sees both the
        percentage and the concrete headcount.
    recall:
        of all true positives, the fraction inside the top k. PU-estimable.
    precision:
        of the top k, the fraction that are true positives. SYNTHETIC-ONLY
        trustworthy (biased downward on real PU data).
    lift:
        precision / base_rate. 1.0 = no better than random; higher is better.
    true_positives_in_k:
        raw count of true positives in the top k (recall * num_true_positives).
    """

    alert_rate: float
    k: int
    recall: float
    precision: float
    lift: float
    true_positives_in_k: int


@dataclass(frozen=True, slots=True)
class SummaryReport:
    """The full fool-proof metric panel for the synthetic PU experiment.

    Two threshold-free headline numbers (computed against true labels):
        pr_auc  -- Average Precision; the PRIMARY ranking metric for imbalanced
                   fraud/AML. Lead with this.
        roc_auc -- shown only with a caveat; inflated under heavy class imbalance
                   and deliberately NOT the headline.
    Plus the capacity sweep (gains rows) and the context needed to read it:
        base_rate            -- num_true_positives / num_accounts; the random
                                baseline lift is measured against.
        human_analyst_precision -- reference SAR-conversion ceiling (~0.45) to
                                draw on the precision curve. A constant, not a
                                model output.
    Nothing here is estimable in production unchanged: precision and lift use the
    synthetic answer key. The formatter states this explicitly.
    """

    pr_auc: float
    roc_auc: float
    rows: tuple[GainsRow, ...]
    base_rate: float
    num_accounts: int
    num_true_positives: int
    human_analyst_precision: float


def gains_table(
    scores: NDArray[np.float64],
    true_label: NDArray[np.int64],
    alert_rates: Sequence[float] = _DEFAULT_ALERT_RATES,
) -> tuple[GainsRow, ...]:
    """Compute recall / precision / lift at each alert-rate cutoff (true labels).

    Sorts scores once and reuses cumulative true-positive counts down the
    ranking, so the whole sweep is O(n log n) regardless of how many rates are
    requested. k for each rate is round(rate * n_total), clamped to [1, n_total].

    scores:      model output per account (higher = more mule-like); shape [M].
    true_label:  1 if truly a mule else 0; shape [M].
    alert_rates: fractions of the population to review (deduped, sorted asc).
                 Each must be in (0, 1].

    Returns one GainsRow per distinct alert_rate. Raises ValueError on shape
    mismatch, out-of-range rate, or a single-class label vector (lift undefined
    when base_rate is 0 or 1).
    """
    if scores.shape != true_label.shape:
        raise ValueError("scores and true_label must have the same shape.")
    if scores.ndim != 1:
        raise ValueError("scores must be 1-D.")

    n_total = _length(true_label)
    n_pos = _count(true_label)
    if n_pos == 0 or n_pos == n_total:
        raise ValueError(
            f"true_label must contain both classes; got {n_pos} positives of {n_total}. "
            + "Lift / precision are undefined when the base rate is 0 or 1."
        )

    rates = tuple(sorted({float(r) for r in alert_rates}))
    if not rates:
        raise ValueError("alert_rates must contain at least one rate.")
    if not all(0.0 < r <= 1.0 for r in rates):
        raise ValueError(f"every alert_rate must be in (0, 1]; got {rates}.")

    base_rate = n_pos / float(n_total)

    order = np.argsort(-scores)
    pos_sorted = true_label[order].astype(np.int64)
    cumulative_pos = np.cumsum(pos_sorted)  # cumulative_pos[t-1] = TPs in top t

    rows: list[GainsRow] = []
    for rate in rates:
        k = int(round(rate * n_total))
        k = max(1, min(k, n_total))
        tp_in_k = _int(cast("np.intp", cumulative_pos[k - 1]))
        recall = tp_in_k / float(n_pos)
        precision = tp_in_k / float(k)
        lift = precision / base_rate  # base_rate > 0 guaranteed above
        rows.append(
            GainsRow(
                alert_rate=rate,
                k=k,
                recall=recall,
                precision=precision,
                lift=lift,
                true_positives_in_k=tp_in_k,
            )
        )
    return tuple(rows)


def evaluate_summary(
    scores: NDArray[np.float64],
    true_label: NDArray[np.int64],
    alert_rates: Sequence[float] = _DEFAULT_ALERT_RATES,
    human_analyst_precision: float = _HUMAN_ANALYST_PRECISION,
) -> SummaryReport:
    """Assemble the full fool-proof panel: PR-AUC + caveated ROC-AUC + sweep.

    scores / true_label:        shape [M]; the synthetic answer key.
    alert_rates:                capacity sweep (see gains_table).
    human_analyst_precision:    reference SAR-conversion ceiling for the chart.

    SYNTHETIC ONLY for the precision/lift parts (true-label answer key). PR-AUC
    and recall are the production-meaningful figures; ROC-AUC is reported only to
    be visibly de-emphasised.
    """
    if scores.shape != true_label.shape:
        raise ValueError("scores and true_label must have the same shape.")
    if scores.ndim != 1:
        raise ValueError("scores must be 1-D.")

    n_total = _length(true_label)
    n_pos = _count(true_label)
    if n_pos == 0 or n_pos == n_total:
        raise ValueError(
            f"true_label must contain both classes; got {n_pos} positives of {n_total}."
        )

    rows = gains_table(scores, true_label, alert_rates)
    return SummaryReport(
        pr_auc=float(average_precision_score(true_label, scores)),
        roc_auc=float(roc_auc_score(true_label, scores)),
        rows=rows,
        base_rate=n_pos / float(n_total),
        num_accounts=n_total,
        num_true_positives=n_pos,
        human_analyst_precision=human_analyst_precision,
    )


def format_summary_report(report: SummaryReport) -> str:
    """Render a SummaryReport as an aligned console block, caveats included."""
    lines: list[str] = []
    lines.append("=" * 72)
    lines.append("MULE-DETECTION EVALUATION  (synthetic; true-label answer key)")
    lines.append("=" * 72)
    lines.append(f"  accounts scored        : {report.num_accounts}")
    lines.append(f"  true mules             : {report.num_true_positives}")
    lines.append(
        f"  base rate              : {report.base_rate:.5f}  "
        + f"({report.base_rate * 100:.3f}% of accounts)"
    )
    lines.append("")
    lines.append("  PRIMARY (threshold-free, robust to imbalance):")
    lines.append(f"    PR-AUC / Avg Precision : {report.pr_auc:.4f}   <- headline")
    lines.append(
        f"    ROC-AUC                : {report.roc_auc:.4f}   "
        + "(de-emphasised: inflated under imbalance; not the headline)"
    )
    lines.append("")
    lines.append("  OPERATING POINTS across review capacity:")
    lines.append("    recall is PU-estimable; PRECISION & LIFT are trustworthy")
    lines.append("    ONLY because this is synthetic (biased on real PU data).")
    lines.append(f"    human-analyst precision reference ~ {report.human_analyst_precision:.2f}")
    lines.append("")
    header = (
        f"  {'alert_rate':>10}  {'k':>7}  {'recall':>8}  "
        + f"{'precision':>10}  {'lift':>8}  {'TP_in_k':>9}"
    )
    lines.append(header)
    for row in report.rows:
        beats = " *" if row.precision >= report.human_analyst_precision else "  "
        lines.append(
            f"  {row.alert_rate * 100:>9.3f}%  {row.k:>7}  {row.recall:>8.4f}  "
            + f"{row.precision:>10.4f}  {row.lift:>7.1f}x  "
            + f"{row.true_positives_in_k:>4}/{report.num_true_positives:<4}{beats}"
        )
    lines.append("")
    lines.append("  ( * = precision at this depth meets/beats the human-analyst")
    lines.append("    reference; the slide-worthy 'matches expert triage' point )")
    lines.append("=" * 72)
    return "\n".join(lines)


# ââ Charting (optional; requires matplotlib) ââ
# Kept in the module so the figure is reproducible from a SummaryReport rather
# than ad-hoc per script. matplotlib is imported INSIDE the function so the rest
# of metrics.py has no hard dependency on it; environments without matplotlib
# can still compute every number above.


def save_summary_chart(
    report: "SummaryReport",
    out_path: str,
    title: str = "Mule detection - gains across review capacity",
    note: str | None = None,
) -> str:
    """Render a two-panel gains/lift chart from a SummaryReport and save to disk.

    Left panel: recall and precision vs. alert rate, with the human-analyst
    precision reference line. Right panel: lift vs. alert rate, with the random
    (1x) baseline. Both x-axes are the alert rate in percent, so the figure is
    read in operational (staffing) terms, not arbitrary k.

    report:   a SummaryReport (use one built from a DENSE alert_rates sweep, e.g.
              numpy.linspace, for smooth curves -- the default 5-point sweep
              produces a coarse plot).
    out_path: file path to write (extension sets the format, e.g. .png/.pdf).
    title:    figure suptitle.
    note:     optional caption drawn under the title; use it to record the data
              provenance (e.g. the run / checkpoint the scores came from). If the
              scores are not real model output, say so here.

    Returns out_path. Raises ImportError (with guidance) if matplotlib is absent.
    Does not fabricate data: every point comes from report.rows.
    """
    try:
        import matplotlib

        matplotlib.use("Agg")  # headless-safe; no display needed
        import matplotlib.pyplot as plt
    except ImportError as exc:  # pragma: no cover - environment dependent
        raise ImportError(
            "save_summary_chart needs matplotlib; install it or compute the "
            "numbers with format_summary_report instead (no plotting required)."
        ) from exc

    if not report.rows:
        raise ValueError("report has no rows to plot.")

    ar = [row.alert_rate * 100.0 for row in report.rows]
    rec = [row.recall for row in report.rows]
    pre = [row.precision for row in report.rows]
    lift = [row.lift for row in report.rows]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    suptitle = title
    fig.suptitle(suptitle, fontsize=11, fontweight="bold")
    if note:
        # small caption line under the suptitle for provenance / caveats
        fig.text(0.5, 0.93, note, ha="center", va="top", fontsize=8, color="#555555")

    ax = axes[0]
    ax.plot(ar, rec, color="#1b5e9c", lw=2.2, label="Recall (detection rate) - PU-estimable")
    ax.plot(ar, pre, color="#c0392b", lw=2.2, label="Precision - synthetic-only")
    ax.axhline(
        report.human_analyst_precision,
        color="#7f8c8d",
        ls="--",
        lw=1.4,
        label=f"Human-analyst precision ~{report.human_analyst_precision:.2f}",
    )
    ax.set_xlabel("Alert rate  (% of accounts reviewed)")
    ax.set_ylabel("Recall / Precision")
    ax.set_title("Recall & precision vs. capacity")
    ax.set_ylim(0.0, 1.0)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, loc="upper right")

    ax = axes[1]
    ax.plot(ar, lift, color="#27772f", lw=2.2)
    ax.axhline(1.0, color="#7f8c8d", ls="--", lw=1.2, label="Random (1x)")
    ax.set_xlabel("Alert rate  (% of accounts reviewed)")
    ax.set_ylabel("Lift  (x better than random)")
    ax.set_title("Lift vs. capacity")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.92 if note else 0.95))
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    return out_path

## 28 · Training loop
`training/loop.py`

`fit`: AdamW, train over `epoch_batches`, evaluate on val each epoch, **early-stop on validation Proxy AUC**, keep the best checkpoint. `TrainConfig` holds the knobs; `EarlyStopper` the patience logic.

In [31]:
from collections.abc import Callable, Iterator, Mapping, Sequence
import copy
import socket
from dataclasses import dataclass
import time
from typing import Protocol, cast

import requests

import numpy as np
from numpy.typing import NDArray
import torch
from torch import Tensor
from torch.nn import Module
from torch.optim import AdamW, Optimizer
from torch_geometric.data import HeteroData
from torch_geometric.typing import EdgeType, NodeType
from tqdm import tqdm


_HAS_PAID: EdgeType = ("Account", "HAS_PAID", "Account")
_ACCOUNT: NodeType = "Account"

BatchTransform = Callable[[HeteroData], HeteroData]


def _identity(batch: HeteroData) -> HeteroData:
    return batch


_TRANSIENT_ERRORS: tuple[type[Exception], ...] = (
    requests.exceptions.ReadTimeout,
    requests.exceptions.ConnectionError,
    requests.exceptions.ChunkedEncodingError,
    requests.exceptions.Timeout,
    TimeoutError,
    ConnectionError,
    socket.timeout,
)
_MAX_BATCH_RETRIES = 5
_RETRY_BACKOFF_S = 10.0


def _resilient_batches(
    make_loader: Callable[[], Iterator[HeteroData]],
) -> Iterator[HeteroData]:
    """Iterate a loader, surviving transient network errors per batch.

    make_loader is a zero-arg factory returning a FRESH batch iterator (each call
    re-issues the underlying TigerGraph queries from the start). When iteration
    raises a transient error mid-stream, we rebuild the loader and fast-forward
    past the batches already yielded, so no batch is reprocessed (which would
    double-count into the optimizer) or skipped (which would drop training data
    or corrupt the validation metric). The re-fetch of skipped batches is wasted
    work, but it is bounded by where the failure occurred and is the price of
    not losing the whole run. A batch that fails _MAX_BATCH_RETRIES times in a
    row re-raises -- that is a real outage, not a blip.

    NOTE: relies on the loader being deterministic across rebuilds (same seed,
    same order), which it is -- make_*_loader uses a fixed seed and shuffle=False.
    """
    delivered = 0
    attempts = 0
    while True:
        produced_this_pass = 0
        try:
            for i, batch in enumerate(make_loader()):
                if i < delivered:
                    continue  # already yielded on a previous pass; skip past it
                yield batch
                delivered += 1
                produced_this_pass += 1
                attempts = 0  # a clean delivery resets the retry budget
            return  # loader exhausted normally -> epoch complete
        except _TRANSIENT_ERRORS as err:
            attempts += 1
            if attempts > _MAX_BATCH_RETRIES:
                raise
            wait = _RETRY_BACKOFF_S * attempts
            print(
                f"    transient fetch error after {delivered} batches "
                + f"(attempt {attempts}/{_MAX_BATCH_RETRIES}); "
                + f"rebuilding loader, retrying in {wait:.0f}s: {type(err).__name__}",
                flush=True,
            )
            time.sleep(wait)


@dataclass(frozen=True, slots=True)
class TrainConfig:
    """
    Training-loop hyperparameters.

    prior:        class prior pi for the nnPU loss (true mule fraction).
    max_epochs:   hard cap on epochs.
    patience:     early-stopping patience (epochs without val-PAUC improvement).
    lr / weight_decay: AdamW settings (weight_decay is the decoupled L2 lever).
    eval_k:       cutoff for precision@k in validation.
    min_delta:    minimum val-PAUC gain to count as a significant improvement.
    positive_weight: weight on the nnPU positive_risk term. None reproduces the
        textbook objective (weight == prior); under extreme imbalance that makes
        the positive term a negligible fraction of the loss, so the optimizer
        drives every logit negative and inverts the ranking. Set it well above
        prior (e.g. 0.5) to give the positives real gradient weight. Passed
        straight through to NonNegativePULoss; see its docstring.
    """

    prior: float
    max_epochs: int = 50
    patience: int = 5
    lr: float = 1e-3
    weight_decay: float = 1e-4
    eval_k: int = 100
    min_delta: float = 1e-4
    positive_weight: float | None = None
    pauc_tie_epsilon: float = 2e-3
    """Half-width of the PAUC 'tie' band for PR-AUC tiebreaking (see EarlyStopper).

    Under heavy class imbalance ROC-AUC (the Proxy AUC) saturates near 1.0, so
    differences between strong epochs at the top end are dominated by validation
    noise rather than real ranking gains -- and selecting on raw PAUC can pin the
    'best' checkpoint to an early epoch that happens to win by noise while having
    worse top-of-ranking behaviour (more false positives). When two epochs are
    within pauc_tie_epsilon of the best PAUC, they are treated as a statistical
    tie on PAUC and broken by PR-AUC (average_precision), which does NOT saturate
    under imbalance and is sensitive to exactly the high-score false positives we
    pay for. PAUC remains the primary, coarse signal; PR-AUC only adjudicates
    near-ties. Set to 0.0 to recover pure-PAUC selection.
    """


class EarlyStopper:
    """
    Tracks the best validation epoch and decides when to stop.

    SELECTION is PAUC-primary with a PR-AUC tiebreaker. Higher is better for
    both (PAUC = validation Proxy AUC; PR-AUC = average_precision). Under heavy
    class imbalance PAUC saturates near 1.0, so among strong epochs its tiny
    differences are mostly validation noise -- and pure-PAUC selection can pin
    the checkpoint to an early epoch that won by noise yet has worse top-of-
    ranking behaviour (more false positives). To fix that WITHOUT abandoning
    PAUC (which PU theory endorses as robust to unlabeled-as-negative label
    corruption), an epoch is taken as the new best-to-save when EITHER:

      * its PAUC exceeds the best PAUC by more than tie_epsilon (a clear PAUC
        win -- decided on PAUC alone, exactly as before), OR
      * its PAUC is within tie_epsilon of the best PAUC (a statistical tie on a
        saturated metric) AND its PR-AUC strictly exceeds the best epoch's
        PR-AUC. PR-AUC does not saturate under imbalance and is sensitive to the
        high-score false positives, so it is the right adjudicator for ties.

    Two judgments remain deliberately separated:

      * is_best (the return of update): the save signal, per the rule above.
      * patience: only a min_delta-significant PAUC gain resets the no-improve
        counter, so stopping behaviour is driven by PAUC alone and is unchanged.
        (The tiebreaker changes WHICH saved checkpoint is best, not WHEN we stop.)
    """

    _patience: int
    _min_delta: float
    _tie_epsilon: float
    _best: float
    _best_secondary: float
    _significant_best: float
    _bad_epochs: int
    best_epoch: int

    def __init__(self, patience: int, min_delta: float, tie_epsilon: float = 0.0) -> None:
        self._patience = patience
        self._min_delta = min_delta
        self._tie_epsilon = tie_epsilon
        self._best = float("-inf")  # best PAUC seen at a saved epoch
        self._best_secondary = float("-inf")  # PR-AUC at the saved best epoch
        self._significant_best = float("-inf")  # PAUC for patience accounting
        self._bad_epochs = 0
        self.best_epoch = -1

    @property
    def best(self) -> float:
        return self._best

    @property
    def best_secondary(self) -> float:
        """PR-AUC recorded at the currently-saved best epoch."""
        return self._best_secondary

    @property
    def bad_epochs(self) -> int:
        """Epochs since the last min_delta-significant PAUC improvement (patience progress)."""
        return self._bad_epochs

    def update(self, score: float, epoch: int, secondary: float = float("-inf")) -> bool:
        """Record an epoch and return True if it is the new best to save.

        score:     primary selection metric (validation Proxy AUC / PAUC).
        secondary: tiebreaker metric (PR-AUC / average_precision); used only when
                   score is within tie_epsilon of the current best PAUC.

        Selection (see class docstring): a clear PAUC win (score > best + eps) is
        best; otherwise a near-tie on PAUC (|score - best| <= eps, i.e.
        score >= best - eps) that strictly improves PR-AUC is also best. Patience
        is managed separately on PAUC, so a sub-delta gain can still be saved
        while the no-improve counter advances toward early stopping.

        With tie_epsilon == 0.0 this reduces to the original strict-PAUC rule
        (the near-tie branch requires score >= best AND better PR-AUC, which only
        improves the secondary at an exact PAUC plateau and never regresses PAUC).
        """
        clear_primary_win = score > self._best + self._tie_epsilon
        within_tie_band = score >= self._best - self._tie_epsilon
        tie_broken_by_secondary = within_tie_band and (secondary > self._best_secondary)

        is_best = clear_primary_win or tie_broken_by_secondary
        if is_best:
            # Track the PAUC of the saved epoch as the comparison point. On a
            # tiebreak the saved PAUC may dip by up to eps; that is the intended
            # trade (a noise-sized PAUC concession for a real PR-AUC gain).
            self._best = max(self._best, score)
            self._best_secondary = max(self._best_secondary, secondary)
            self.best_epoch = epoch

        # patience: PAUC-only, unchanged from the original.
        if score > self._significant_best + self._min_delta:
            self._significant_best = score
            self._bad_epochs = 0
        else:
            self._bad_epochs += 1
        return is_best

    @property
    def should_stop(self) -> bool:
        return self._bad_epochs >= self._patience


class _NodeStore(Protocol):
    n_id: Tensor
    x: Tensor


class _EdgeStore(Protocol):
    edge_index: Tensor
    edge_attr: Tensor


def _node_store(batch: HeteroData, key: NodeType) -> _NodeStore:
    return cast(_NodeStore, cast(object, batch[key]))


def _edge_store(batch: HeteroData, key: EdgeType) -> _EdgeStore:
    return cast(_EdgeStore, cast(object, batch[key]))


def _forward(model: Module, batch: HeteroData, device: torch.device) -> Tensor:
    x_dict: dict[NodeType, Tensor] = {_ACCOUNT: _node_store(batch, _ACCOUNT).x.to(device)}
    node_counts: dict[NodeType, int] = {
        ntype: int(_node_store(batch, ntype).n_id.shape[0]) for ntype in batch.node_types
    }
    edge_index_dict: dict[EdgeType, Tensor] = {
        etype: _edge_store(batch, etype).edge_index.to(device) for etype in batch.edge_types
    }
    edge_attr_dict: dict[EdgeType, Tensor] = {
        _HAS_PAID: _edge_store(batch, _HAS_PAID).edge_attr.to(device)
    }
    return cast(Tensor, model(x_dict, edge_index_dict, edge_attr_dict, node_counts))


def _seed_ids(batch: HeteroData, mapper_to_ids: Callable[[list[int]], list[str]]) -> list[str]:
    bsize = int(batch[_ACCOUNT].batch_size)  # pyright: ignore[reportAny]
    seed_int = cast(list[int], _node_store(batch, _ACCOUNT).n_id[:bsize].tolist())
    return mapper_to_ids(seed_int)


def _targets(seed_ids: Sequence[str], label_of: Mapping[str, int], device: torch.device) -> Tensor:
    return torch.tensor([label_of[s] for s in seed_ids], dtype=torch.long, device=device)


def train_epoch(
    model: Module,
    make_loader: Callable[[], Iterator[HeteroData]],
    loss_fn: NonNegativePULoss,
    optimizer: Optimizer,
    pu_label_of: Mapping[str, int],
    mapper_to_ids: Callable[[list[int]], list[str]],
    batch_transform: BatchTransform = _identity,
    total_batches: int | None = None,
    epoch: int = 0,
    device: torch.device | None = None,
) -> float:
    """
    One training epoch. Returns the mean training loss over batches.

    For each batch: forward the whole sampled neighborhood, slice to the seed
    accounts (logits[:batch_size]), look up their pu_label targets, compute the
    nnPU loss on the seeds only, and backprop. Neighbors only inform embeddings;
    they never contribute to the loss.

    total_batches, if given, drives the progress bar's ETA (the loader is a lazy
    generator with no len, so the count must be passed in). miniters=1 because
    per-batch latency is erratic (each batch is a TigerGraph round-trip).
    """
    dev = device if device is not None else torch.device("cpu")
    _ = model.train()
    total = 0.0
    count = 0
    bar = tqdm(
        _resilient_batches(make_loader),
        total=total_batches,
        unit="batch",
        miniters=1,
        desc=f"train e{epoch}",
    )
    for raw in bar:
        batch = batch_transform(raw)
        logits = _forward(model, batch, dev)
        seeds = _seed_ids(batch, mapper_to_ids)
        seed_logits = logits[: len(seeds)]
        targets = _targets(seeds, pu_label_of, dev)

        train_loss, _ = cast("tuple[Tensor, Tensor]", loss_fn(seed_logits, targets))
        optimizer.zero_grad()
        _ = train_loss.backward()
        _ = optimizer.step()

        total += float(train_loss.item())
        count += 1
        bar.set_postfix(loss=f"{total / count:.4f}")
    return total / max(count, 1)


def validate(
    model: Module,
    make_loader: Callable[[], Iterator[HeteroData]],
    pu_label_of: Mapping[str, int],
    mapper_to_ids: Callable[[list[int]], list[str]],
    eval_k: int,
    total_batches: int | None = None,
    epoch: int = 0,
    device: torch.device | None = None,
) -> ValScores:
    """
    Score the model over the validation seeds and return ranking metrics.

    Collects per-seed scores (sigmoid of the logit) and the seed's pu_label
    across all val batches, then evaluates ranking quality against pu_label.
    This uses only the labels the model also trains on, so it is production-valid.
    The loader must use the
    validation sampling regime (allow_val=True, allow_test=False) so no test
    account leaks into a val neighborhood.
    """
    dev = device if device is not None else torch.device("cpu")
    _ = model.eval()
    scores: list[float] = []
    labels: list[int] = []
    raw_logits: list[float] = []
    bar = tqdm(
        _resilient_batches(make_loader),
        total=total_batches,
        unit="batch",
        miniters=1,
        desc=f"  val e{epoch}",
    )
    with torch.no_grad():
        for batch in bar:
            logits = _forward(model, batch, dev)
            seeds = _seed_ids(batch, mapper_to_ids)
            seed_logits = logits[: len(seeds)]
            probs = cast(list[float], torch.sigmoid(seed_logits).tolist())
            scores.extend(probs)
            raw_logits.extend(cast(list[float], seed_logits.tolist()))
            labels.extend(pu_label_of[s] for s in seeds)

    scores_arr: NDArray[np.float64] = np.asarray(scores, dtype=np.float64)
    labels_arr: NDArray[np.int64] = np.asarray(labels, dtype=np.int64)

    logits_arr: NDArray[np.float64] = np.asarray(raw_logits, dtype=np.float64)
    pos_mask = labels_arr == 1
    pos_logits = logits_arr[pos_mask]
    unl_logits = logits_arr[~pos_mask]
    pos_mean = float(pos_logits.mean()) if pos_logits.size else float("nan")
    unl_mean = float(unl_logits.mean()) if unl_logits.size else float("nan")
    print(
        f"    logits: all[mean={logits_arr.mean():.3f} std={logits_arr.std():.3f} "
        + f"min={logits_arr.min():.3f} max={logits_arr.max():.3f}] "
        + f"pos_mean={pos_mean:.3f} unl_mean={unl_mean:.3f} "
        + f"separation={pos_mean - unl_mean:+.3f}",
        flush=True,
    )

    return evaluate_ranking(scores_arr, labels_arr, k=eval_k)


@dataclass(frozen=True, slots=True)
class EpochReport:
    epoch: int
    train_loss: float
    val_average_precision: float
    val_precision_at_k: float
    val_roc_auc: float
    is_best: bool


@dataclass(frozen=True, slots=True)
class FitResult:
    """Outcome of a training run.

    reports holds the per-epoch history. best_state_dict is a deep copy of the
    model weights at the epoch with the highest validation Proxy AUC (PAUC, the
    PU model-selection signal; deep-copied so later epochs do not mutate it), and
    best_epoch / best_val_pauc identify that epoch. After fit returns, the model
    has these best weights loaded, so it is ready to score or to save directly.
    """

    reports: list[EpochReport]
    best_state_dict: dict[str, Tensor]
    best_epoch: int
    best_val_pauc: float


def fit(
    model: Module,
    make_train_loader: Callable[[], Iterator[HeteroData]],
    make_val_loader: Callable[[], Iterator[HeteroData]],
    pu_label_of: Mapping[str, int],
    mapper_to_ids: Callable[[list[int]], list[str]],
    config: TrainConfig,
    optimizer: Optimizer | None = None,
    batch_transform: BatchTransform = _identity,
    train_batches: int | None = None,
    val_batches: int | None = None,
    device: torch.device | None = None,
    on_best: Callable[[dict[str, Tensor], int, float], None] | None = None,
) -> FitResult:
    """
    Train with early stopping on validation Proxy AUC (PAUC).

    make_train_loader / make_val_loader are zero-arg factories returning a fresh
    iterator of HeteroData batches for one epoch (fresh so each epoch reshuffles
    seeds and resamples neighborhoods). The train loader must use the training
    sampling regime (allow_val=False, allow_test=False); the val loader must use
    allow_val=True, allow_test=False. Returns one EpochReport per epoch run.

    train_batches / val_batches are the per-epoch batch counts, passed through to
    the progress bars for ETA (the loaders are lazy generators with no len).

    on_best, if given, is called the moment a new best epoch is found, with the
    best weights, epoch index, and val PAUC. Use it to persist the best
    checkpoint DURING training, so an interrupted long run still leaves the best
    model on disk rather than losing everything (the loop only returns at the
    very end).

    The optimizer defaults to AdamW(lr, weight_decay) over the model parameters;
    pass a pre-built optimizer to override.
    """
    dev = device if device is not None else select_device()
    _ = model.to(dev)
    opt: Optimizer = (
        optimizer
        if optimizer is not None
        else AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    )
    loss_fn = NonNegativePULoss(prior=config.prior, positive_weight=config.positive_weight)
    stopper = EarlyStopper(
        patience=config.patience,
        min_delta=config.min_delta,
        tie_epsilon=config.pauc_tie_epsilon,
    )

    reports: list[EpochReport] = []
    best_state: dict[str, Tensor] = copy.deepcopy(model.state_dict())
    best_epoch = -1
    best_val_pauc = float("-inf")
    for epoch in range(config.max_epochs):
        train_loss = train_epoch(
            model,
            make_train_loader,
            loss_fn,
            opt,
            pu_label_of,
            mapper_to_ids,
            batch_transform,
            total_batches=train_batches,
            epoch=epoch,
            device=dev,
        )
        val = validate(
            model,
            make_val_loader,
            pu_label_of,
            mapper_to_ids,
            config.eval_k,
            total_batches=val_batches,
            epoch=epoch,
            device=dev,
        )
        prev_best_pauc = stopper.best
        prev_best_ap = stopper.best_secondary
        is_best = stopper.update(val.roc_auc, epoch, secondary=val.average_precision)
        if is_best:
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            best_val_pauc = val.roc_auc
            if on_best is not None:
                on_best(best_state, best_epoch, best_val_pauc)
        reports.append(
            EpochReport(
                epoch=epoch,
                train_loss=train_loss,
                val_average_precision=val.average_precision,
                val_precision_at_k=val.precision_at_k,
                val_roc_auc=val.roc_auc,
                is_best=is_best,
            )
        )

        if is_best:
            # Was this a clear PAUC win, or a PR-AUC tiebreak within the PAUC
            # noise band? Classify from the pre-update bests for a transparent log.
            if val.roc_auc > prev_best_pauc + config.pauc_tie_epsilon:
                marker = " *best (PAUC)"
            else:
                marker = (
                    f" *best (PR-AUC tiebreak: PAUC {val.roc_auc:.4f}~"
                    + f"{prev_best_pauc:.4f}, AP {val.average_precision:.4f}>"
                    + f"{prev_best_ap:.4f})"
                )
        else:
            marker = f"  (no gain for {stopper.bad_epochs})"
        print(
            f"epoch {epoch:>2} | train_loss {train_loss:.4f} | "
            + f"val PAUC {val.roc_auc:.4f} | "
            + f"AP {val.average_precision:.4f} | "
            + f"P@{val.k} {val.precision_at_k:.4f} | "
            + f"pos {val.num_labeled_positives}/{val.num_evaluated}{marker}",
            flush=True,
        )

        if stopper.should_stop:
            print(
                f"early stop: val PAUC has not improved for {config.patience} epochs "
                + f"(best was epoch {best_epoch}, PAUC {best_val_pauc:.4f}).",
                flush=True,
            )
            break

    _ = model.load_state_dict(best_state)
    return FitResult(
        reports=reports,
        best_state_dict=best_state,
        best_epoch=best_epoch,
        best_val_pauc=best_val_pauc,
    )

# Part 2 · Running it

Everything above is the library. Now we **use** it against the live graph: connect, pull one real batch, push it through the model, and take one nnPU training step. Every shape printed here should match the diagrams from the slides — `x = [N, 31]`, `edge_attr = [E, 34]`, six edge types, one logit per account.

## D1 · Connect
Read `.env`, open the TigerGraph client. (The schema, Entity Resolution → `Same_As`, and WCC → `Connected_Component` are shown live in GraphStudio.)

In [ ]:
client = Client(Settings())
print('connected:', client.graphname)

## D2 · Derive the temporal spec from the graph
`max_bins` and the reference epoch come from the data, so `edge_dim` is derived, not hardcoded. Each call is a full edge scan — expect a pause.

In [ ]:
spec = derive_temporal_spec(client)
REF_EPOCH_S = derive_reference_epoch_s(client)
MAX_BINS = spec.max_bins
print(f'max_bins = {MAX_BINS}  ->  edge_dim = {spec.edge_dim}  (flat_edge_dim({MAX_BINS}) = {flat_edge_dim(MAX_BINS)})')
print(f'reference_epoch_s = {REF_EPOCH_S:.0f}')

## D3 · One real batch  →  HeteroData
Build the backend (it owns the shared client + mapper) and pull a single batch. **One in-database sample per batch**, not per account. The seeds are positive-anchored via `epoch_batches`, the same construction training uses — so the batch contains known mules, not just unlabelled accounts.

In [ ]:
# positive-anchored seeds (same machinery as the training loop)
train_seeds = fetch_split_seeds(client, 'train')
train_pool = SeedPool(
    positives=tuple(a for a, y in train_seeds.pu_label_of.items() if y == 1),
    unlabeled=tuple(a for a, y in train_seeds.pu_label_of.items() if y == 0),
)
print(f'SeedPool: {train_pool.num_positives} positives, {train_pool.num_unlabeled} unlabeled')

BATCH_SIZE = 8
POSITIVES_PER_BATCH = 2          # demo ratio; the loop uses ~32-64 on full batches
first_batch_seeds = next(epoch_batches(train_pool, batch_size=BATCH_SIZE,
                                       positives_per_batch=POSITIVES_PER_BATCH, seed=1337))
seed_pu = torch.tensor([train_seeds.pu_label_of[a] for a in first_batch_seeds], dtype=torch.long)
print(f'batch seeds : {list(first_batch_seeds)}')
print(f'pu_labels   : {seed_pu.tolist()}  ({int(seed_pu.sum())} positive, {int((seed_pu==0).sum())} unlabeled)')
assert int(seed_pu.sum()) >= 1, 'expected >=1 positive in the batch'

backend = TigerGraphRemoteBackend(client)
loader = backend.make_loader(
    seed_ids=first_batch_seeds, reference_epoch_s=REF_EPOCH_S, max_bins=MAX_BINS,
    batch_size=BATCH_SIZE, shuffle=False, allow_val=False, allow_test=False,
)
print('\npulling one batch (server sampling - slow)...')
batch = cast(HeteroData, next(iter(loader)))
print('done.')
print('node types :', batch.node_types)
print('edge types :', batch.edge_types)

## D4 · The batch's shapes
Node features filled by the FeatureStore (`Account.x = [N, 31]`; featureless types get learned embeddings in the model), HAS_PAID `edge_attr = [E, 34]` filled by the loader transform, structure in every relation's `edge_index`.

In [ ]:
_HAS_PAID = ('Account', 'HAS_PAID', 'Account')
for nt in batch.node_types:
    s = batch[nt]
    xs = tuple(s.x.shape) if hasattr(s, 'x') else None
    print(f'  {nt:16s}: {int(s.n_id.shape[0]):4d} nodes   x={xs}')
print()
for et in batch.edge_types:
    s = batch[et]
    eas = tuple(s.edge_attr.shape) if hasattr(s, 'edge_attr') else None
    print(f'  {str(et):52s}: {int(s.edge_index.shape[1]):4d} edges   edge_attr={eas}')
hp = batch[_HAS_PAID]
print(f'\nHAS_PAID edge_attr width == flat_edge_dim({MAX_BINS}) == {flat_edge_dim(MAX_BINS)}: '
      f'{hp.edge_attr.shape[1] == flat_edge_dim(MAX_BINS)}')

## D5 · One forward pass  →  logits
Repackage the batch into the dicts the model takes, build `MulePatternModel`, run a forward pass. Output is **one mule-likelihood logit per account**.

In [ ]:
x_dict = {'Account': batch['Account'].x}
node_counts = {nt: int(batch[nt].n_id.shape[0]) for nt in batch.node_types}
edge_index_dict = {et: batch[et].edge_index for et in batch.edge_types}
edge_attr_dict = {_HAS_PAID: batch[_HAS_PAID].edge_attr}

model = MulePatternModel(account_in_dim=NUM_ACCOUNT_FEATURES,
                         edge_dim=flat_edge_dim(MAX_BINS), hidden_dim=64, heads=4)
model.eval()
with torch.no_grad():
    logits = cast(Tensor, model(x_dict, edge_index_dict, edge_attr_dict, node_counts))
n_acct = node_counts['Account']
print(f'logits shape : {tuple(logits.shape)}  (expect [{n_acct}])')
print(f'one per account: {int(logits.shape[0]) == n_acct}')
print(f'first logits : {[round(v, 4) for v in logits[:5].tolist()]}')

## D6 · One nnPU training step
Loss on the **seed** accounts (first `b` rows). Because the batch is positive-anchored, the positive-risk term is active. One backward + optimizer step changes the readout weights — that's one step of what `fit` repeats over the whole training run, early-stopping on validation Proxy AUC.

In [ ]:
opt = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = NonNegativePULoss(prior=0.003, positive_weight=0.5)
bsz = int(getattr(batch, 'batch_size', len(first_batch_seeds)))

w_before = model._head.weight.detach().clone()
model.train()
opt.zero_grad()
logits = cast(Tensor, model(x_dict, edge_index_dict, edge_attr_dict, node_counts))
train_loss, objective = loss_fn(logits[:bsz], seed_pu)
train_loss.backward()
opt.step()
w_after = model._head.weight.detach()
print(f'train_loss : {train_loss.item():.6f}')
print(f'objective  : {objective.item():.6f}')
print(f'positive term active (>=1 positive in batch): {int(seed_pu.sum()) >= 1}')
print(f'readout weights changed after step: {not torch.equal(w_before, w_after)}')
print(f'  max |delta|: {(w_after - w_before).abs().max().item():.3e}')

## D7 · The non-negative correction, made visible
On one untrained step the clamp usually does **not** fire (it triggers only when the estimated negative risk goes below zero — an overfitting signal). To *see* it, force the regime: positives scored high, unlabelled scored low, with a larger prior. The loss then switches to push the negative risk back up — that flip is the nnPU correction.

In [ ]:
clamp_fn = NonNegativePULoss(prior=0.3, positive_weight=0.5)
clamp_logits = torch.tensor([6., -6., -6., 6., -6., -6., -6., -6., -6., -6.])
clamp_targets = torch.tensor([1, 0, 0, 1, 0, 0, 0, 0, 0, 0])
tl, obj = clamp_fn(clamp_logits, clamp_targets)
print(f'train_loss : {tl.item():.6f}')
print(f'objective  : {obj.item():.6f}')
print(f'correction fired (train_loss != objective): {abs(tl.item() - obj.item()) > 1e-9}')
print('  -> objective went negative; backprop switches to push negative_risk back toward 0.')